## Notebook 02 — CIC-IDS-2017: Classical ML Models + SHAP/LIME Analysis

**Models:** Random Forest, Extra Trees, Decision Tree, XGBoost, LightGBM, CatBoost, Logistic Regression  
**Dataset:** CIC-IDS-2017 (12 classes, ~79 K samples)  
**Steps:** Data loading → Model training (10 seeds × 5 folds) → SHAP/LIME computation → RQ1–RQ5 evaluation  

> Run the **Configuration** cell below before any other cell to set data paths.


In [ ]:
from pathlib import Path

# ── DATA PATHS — adjust to your local setup ──────────────────────────────────
# Point BASE_DIR to the folder that contains your checkpoint sub-directories.
# Expected structure:
#   BASE_DIR/
#     edgeiiot_checkpoints/   <- Edge-IIoT intermediate results
#     cicids2017_checkpoints/ <- CIC-IDS-2017 intermediate results
#     dnn_lstm_checkpoints/   <- DNN / LSTM intermediate results
#     ML-EdgeIIoT-dataset.csv <- raw Edge-IIoT CSV
BASE_DIR       = Path("../data")
EDGEIIOT_DIR   = BASE_DIR / "edgeiiot_checkpoints"
CICIDS2017_DIR = BASE_DIR / "cicids2017_checkpoints"
DNN_LSTM_DIR   = BASE_DIR / "dnn_lstm_checkpoints"
FIGURES_DIR    = Path("../figures")
RESULTS_DIR    = BASE_DIR / "analiz_sonuclari"
# ─────────────────────────────────────────────────────────────────────────────


In [ ]:
# ============================================================
# CIC-IDS-2017 — IDS + XAI Analizi
# Edge-IIoT yapısıyla birebir uyumlu
# ============================================================

import pandas as pd
import numpy as np
import os, pickle, psutil, glob, warnings
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
warnings.filterwarnings('ignore')

# ── Checkpoint fonksiyonları ──────────────────────────────────────────────
CHECKPOINT_DIR = r"str(CICIDS2017_DIR)"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

def yukle(dosya_adi):
    yol = os.path.join(CHECKPOINT_DIR, dosya_adi)
    with open(yol, 'rb') as f:
        return pickle.load(f)

def kaydet(nesne, dosya_adi):
    yol = os.path.join(CHECKPOINT_DIR, dosya_adi)
    with open(yol, 'wb') as f:
        pickle.dump(nesne, f)
    print(f"  💾 Kaydedildi: {dosya_adi}")

def kontrol_et(dosya_adi):
    return os.path.exists(os.path.join(CHECKPOINT_DIR, dosya_adi))

def ram_goster(etiket=""):
    ram = psutil.virtual_memory()
    print(f"  🧠 RAM [{etiket}]: {ram.used/1e9:.1f} GB / {ram.total/1e9:.1f} GB")

# ── Veri yükleme ──────────────────────────────────────────────────────────
# !! Buraya kendi klasör yolunu yaz !!
VERI_KLASORU   = r"str(CICIDS2017_DIR)"

if kontrol_et("veri_cicids.pkl"):
    print("♻️  Veri checkpoint'ten yükleniyor...")
    veri = yukle("veri_cicids.pkl")
    X_scaled      = veri['X_scaled']
    y             = veri['y']
    sinif_isimleri = veri['sinif_isimleri']
    feature_names  = veri['feature_names']
    print(f"✅ Veri yüklendi: {X_scaled.shape}, {len(np.unique(y))} sınıf")

else:
    print("📂 CSV dosyaları okunuyor...")

    # Tüm CSV'leri birleştir
    csv_listesi = glob.glob(os.path.join(VERI_KLASORU, "*.csv"))
    print(f"  {len(csv_listesi)} dosya bulundu: {[os.path.basename(f) for f in csv_listesi]}")

    parcalar = []
    for csv_yol in csv_listesi:
        try:
            df_parca = pd.read_csv(csv_yol, encoding='utf-8', low_memory=False)
            parcalar.append(df_parca)
            print(f"  ✅ {os.path.basename(csv_yol)}: {df_parca.shape}")
        except Exception as e:
            print(f"  ❌ {os.path.basename(csv_yol)}: {e}")

    df = pd.concat(parcalar, ignore_index=True)
    print(f"\n📊 Toplam: {df.shape}")

    # Sütun adlarını temizle (baştaki/sondaki boşluklar)
    df.columns = df.columns.str.strip()

    # Label sütununu bul
    label_sutun = None
    for sut in df.columns:
        if sut.strip().lower() in ['label', 'labels', ' label']:
            label_sutun = sut
            break
    print(f"  Label sütunu: '{label_sutun}'")
    df[label_sutun] = df[label_sutun].str.strip()

    # Sınıf dağılımı
    print("\n📊 Sınıf dağılımı (ham):")
    print(df[label_sutun].value_counts())

    # ── Temizlik ──────────────────────────────────────────────────────────
    # Inf ve NaN temizliği
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    nan_once = df.isnull().sum().sum()
    print(f"\n  NaN sayısı: {nan_once}")
    df.dropna(inplace=True)
    print(f"  NaN temizliği sonrası: {df.shape}")

    # Sayısal olmayan özellik sütunlarını çıkar
    ozellik_sutunlari = [c for c in df.columns if c != label_sutun]
    df_X = df[ozellik_sutunlari].select_dtypes(include=[np.number])
    feature_names = list(df_X.columns)
    print(f"  Sayısal özellik sayısı: {len(feature_names)}")

    # ── Örnekleme (her sınıftan max 10.000) ──────────────────────────────
    # CIC-IDS2017'de BENIGN çok fazla, dengelemek için cap koy
    MAX_SINIF = 10_000
    df['__label__'] = df[label_sutun]
    df_ornekler = []
    for sinif in df['__label__'].unique():
        df_sinif = df[df['__label__'] == sinif]
        if len(df_sinif) > MAX_SINIF:
            df_sinif = df_sinif.sample(MAX_SINIF, random_state=42)
        df_ornekler.append(df_sinif)
    df = pd.concat(df_ornekler, ignore_index=True)
    print(f"\n✅ Örnekleme sonrası: {df.shape}")
    print(df['__label__'].value_counts())

    # Az örnekli sınıfları çıkar (< 50 örnek — genelde 'Heartbleed', 'Infiltration')
    MIN_ORNEK = 50
    sinif_sayilari = df['__label__'].value_counts()
    gecerli_siniflar = sinif_sayilari[sinif_sayilari >= MIN_ORNEK].index
    df = df[df['__label__'].isin(gecerli_siniflar)]
    print(f"\n✅ Az örnekli sınıf temizliği sonrası: {df.shape}")
    print(f"  Kalan sınıflar: {sorted(df['__label__'].unique())}")

    # ── Label encoding ────────────────────────────────────────────────────
    le = LabelEncoder()
    y = le.fit_transform(df['__label__'])
    sinif_isimleri = list(le.classes_)
    print(f"\n  Sınıf listesi: {sinif_isimleri}")

    # ── Özellik matrisi ve ölçekleme ──────────────────────────────────────
    X = df[feature_names].values.astype(np.float32)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    print(f"\n✅ Final: X={X_scaled.shape}, y={y.shape}, sınıf={len(sinif_isimleri)}")

    # Checkpoint kaydet
    kaydet({
        'X_scaled'      : X_scaled,
        'y'             : y,
        'sinif_isimleri': sinif_isimleri,
        'feature_names' : feature_names,
        'scaler'        : scaler,
        'le'            : le,
    }, "veri_cicids.pkl")

# ── Model tanımları ───────────────────────────────────────────────────────
def get_modeller(seed):
    return [
        ('RandomForest', RandomForestClassifier(
            n_estimators=200, max_depth=20, min_samples_leaf=2,
            class_weight='balanced', random_state=seed, n_jobs=-1)),
        ('ExtraTrees', ExtraTreesClassifier(
            n_estimators=200, max_depth=20, min_samples_leaf=2,
            class_weight='balanced', random_state=seed, n_jobs=-1)),
        ('DecisionTree', DecisionTreeClassifier(
            max_depth=20, min_samples_leaf=2,
            class_weight='balanced', random_state=seed)),
        ('XGBoost', XGBClassifier(
            n_estimators=200, max_depth=6, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            random_state=seed, eval_metric='mlogloss', verbosity=0,
            n_jobs=-1)),
        ('LightGBM', LGBMClassifier(
            n_estimators=300, num_leaves=63, learning_rate=0.05,
            min_child_samples=10, class_weight='balanced',
            random_state=seed, verbose=-1, n_jobs=-1)),
        ('CatBoost', CatBoostClassifier(
            iterations=200, depth=8, learning_rate=0.05,
            random_seed=seed, verbose=0,
            auto_class_weights='Balanced')),
        ('LogisticRegression', LogisticRegression(
            max_iter=2000, C=0.1, class_weight='balanced',
            solver='saga', random_state=seed, n_jobs=-1,
            multi_class='multinomial')),
    ]

# ── Eğitim döngüsü ────────────────────────────────────────────────────────
SEEDLER    = list(range(10))
N_FOLD     = 5
X_arr      = X_scaled
y_arr      = y

print("\n🚀 Model eğitimi başlıyor (10 seed × 5 fold = 350 model)...\n")

for seed in SEEDLER:
    dosya = f"modeller_cicids_seed{seed}.pkl"

    if kontrol_et(dosya):
        print(f"  ♻️  Seed {seed} zaten mevcut, atlanıyor...")
        continue

    print(f"  🔄 Seed {seed} eğitiliyor...")
    ram_goster(f"seed {seed}")

    skf           = StratifiedKFold(n_splits=N_FOLD, shuffle=True, random_state=seed)
    seed_sonuclar = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X_arr, y_arr)):
        X_train, X_test = X_arr[train_idx], X_arr[test_idx]
        y_train, y_test = y_arr[train_idx], y_arr[test_idx]

        for model_adi, model in get_modeller(seed):
            model.fit(X_train, y_train)
            y_pred   = model.predict(X_test)
            f1       = f1_score(y_test, y_pred, average='macro')
            accuracy = accuracy_score(y_test, y_pred)

            seed_sonuclar.append({
                'model_adi': model_adi,
                'seed'     : seed,
                'fold'     : fold,
                'f1_macro' : round(f1, 4),
                'accuracy' : round(accuracy, 4),
                'model_obj': model,
                'X_test'   : X_test,
                'y_test'   : y_test,
            })

    kaydet(seed_sonuclar, dosya)
    print(f"  ✅ Seed {seed} tamamlandı")
    ram_goster(f"seed {seed} sonrası")

# ── Performans tablosu ────────────────────────────────────────────────────
MODEL_SIRASI = [
    'RandomForest', 'ExtraTrees', 'DecisionTree',
    'XGBoost', 'LightGBM', 'CatBoost', 'LogisticRegression'
]

tum_sonuclar = []
for seed in SEEDLER:
    tum_sonuclar.extend(yukle(f"modeller_cicids_seed{seed}.pkl"))

df_sonuc = pd.DataFrame([
    {k: v for k, v in s.items() if k not in ('model_obj', 'X_test', 'y_test')}
    for s in tum_sonuclar
])
kaydet(df_sonuc, "df_sonuc_cicids.pkl")

print("\n✅ Final Performans Tablosu (CIC-IDS-2017):")
print(f"\n{'Model':<22} {'F1 Macro':>12}   {'Accuracy':>12}")
print("-" * 52)
for m in MODEL_SIRASI:
    alt = df_sonuc[df_sonuc['model_adi'] == m]
    f1  = alt['f1_macro'].mean()
    acc = alt['accuracy'].mean()
    f1s = alt['f1_macro'].std()
    acs = alt['accuracy'].std()
    print(f"  {m:<20} {f1:.4f} ± {f1s:.4f}   {acc:.4f} ± {acs:.4f}")

In [ ]:
# ============================================================
# CIC-IDS-2017 — SHAP + LIME Hesaplama
# ============================================================

import shap
import lime
import lime.lime_tabular
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
warnings.filterwarnings('ignore')

# ── Parametreler ──────────────────────────────────────────────────────────
N_ORNEK       = 500    # XAI hesabı için örnek sayısı
N_TEKRAR      = 10     # Her örnek için tekrar (pertürbasyon için)
RANDOM_STATE  = 42

MODEL_SIRASI = [
    'RandomForest', 'ExtraTrees', 'DecisionTree',
    'XGBoost', 'LightGBM', 'CatBoost', 'LogisticRegression'
]

# ── Seed 0'dan referans modelleri ve test setini al ───────────────────────
seed0_sonuclar = yukle("modeller_cicids_seed0.pkl")

# Her model için fold 0'ı al
ref_modeller = {}
ref_X_test   = None
ref_y_test   = None

for kayit in seed0_sonuclar:
    if kayit['fold'] == 0:
        ref_modeller[kayit['model_adi']] = kayit['model_obj']
        if ref_X_test is None:
            ref_X_test = kayit['X_test']
            ref_y_test = kayit['y_test']

print(f"✅ Referans modeller yüklendi: {list(ref_modeller.keys())}")
print(f"✅ Test seti: {ref_X_test.shape}")

# ── 500 örnek seç (stratified) ────────────────────────────────────────────
np.random.seed(RANDOM_STATE)
from sklearn.model_selection import StratifiedShuffleSplit

if len(ref_X_test) > N_ORNEK:
    sss = StratifiedShuffleSplit(n_splits=1, test_size=N_ORNEK, random_state=RANDOM_STATE)
    _, ornek_idx = next(sss.split(ref_X_test, ref_y_test))
else:
    ornek_idx = np.arange(len(ref_X_test))

X_ornek = ref_X_test[ornek_idx]
y_ornek = ref_y_test[ornek_idx]
print(f"✅ XAI örnek seti: {X_ornek.shape}")

# ── SHAP Değerleri ────────────────────────────────────────────────────────
print("\n🔄 SHAP hesaplanıyor...")

shap_degerleri = {}

for model_adi, model in ref_modeller.items():
    if kontrol_et(f"shap_cicids_{model_adi}.pkl"):
        print(f"  ♻️  {model_adi} SHAP zaten mevcut")
        shap_degerleri[model_adi] = yukle(f"shap_cicids_{model_adi}.pkl")
        continue

    print(f"  🔄 {model_adi} SHAP hesaplanıyor...")

    try:
        if model_adi in ['RandomForest', 'ExtraTrees', 'DecisionTree']:
            explainer = shap.TreeExplainer(model)
            shap_vals = explainer.shap_values(X_ornek)
            # Multiclass: liste → ortalama abs değer
            if isinstance(shap_vals, list):
                shap_abs = np.mean([np.abs(sv) for sv in shap_vals], axis=0)
            else:
                shap_abs = np.abs(shap_vals)

        elif model_adi in ['XGBoost', 'LightGBM', 'CatBoost']:
            explainer = shap.TreeExplainer(model)
            shap_vals = explainer.shap_values(X_ornek)
            if isinstance(shap_vals, list):
                shap_abs = np.mean([np.abs(sv) for sv in shap_vals], axis=0)
            else:
                shap_abs = np.abs(shap_vals)

        elif model_adi == 'LogisticRegression':
            explainer = shap.LinearExplainer(model, X_ornek)
            shap_vals = explainer.shap_values(X_ornek)
            if isinstance(shap_vals, list):
                shap_abs = np.mean([np.abs(sv) for sv in shap_vals], axis=0)
            else:
                shap_abs = np.abs(shap_vals)

        kaydet(shap_abs, f"shap_cicids_{model_adi}.pkl")
        shap_degerleri[model_adi] = shap_abs
        print(f"  ✅ {model_adi} SHAP tamamlandı: {shap_abs.shape}")

    except Exception as e:
        print(f"  ❌ {model_adi} SHAP hatası: {e}")

# ── LIME Değerleri ────────────────────────────────────────────────────────
print("\n🔄 LIME hesaplanıyor...")

# LIME explainer — tüm modeller için ortak
lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data  = X_ornek,
    feature_names  = feature_names,
    class_names    = sinif_isimleri,
    mode           = 'classification',
    random_state   = RANDOM_STATE,
    discretize_continuous = True
)

lime_degerleri = {}
N_LIME_ORNEK = 100  # LIME yavaş, 100 örnek yeterli

for model_adi, model in ref_modeller.items():
    if kontrol_et(f"lime_cicids_{model_adi}.pkl"):
        print(f"  ♻️  {model_adi} LIME zaten mevcut")
        lime_degerleri[model_adi] = yukle(f"lime_cicids_{model_adi}.pkl")
        continue

    print(f"  🔄 {model_adi} LIME hesaplanıyor (100 örnek)...")

    lime_mat = np.zeros((N_LIME_ORNEK, len(feature_names)))

    for i in range(N_LIME_ORNEK):
        try:
            exp = lime_explainer.explain_instance(
                X_ornek[i],
                model.predict_proba,
                num_features = len(feature_names),
                num_samples  = 1000,
                top_labels   = 1
            )
            label = exp.available_labels()[0]
            lime_map = dict(exp.as_list(label=label))
            for j, feat in enumerate(feature_names):
                # LIME özellik adları bazen discretize edilmiş olur
                for key, val in lime_map.items():
                    if feat in key:
                        lime_mat[i, j] = abs(val)
                        break
        except Exception as e:
            pass  # Hata olan örneği 0 bırak

        if (i + 1) % 20 == 0:
            print(f"    {i+1}/{N_LIME_ORNEK} örnek tamamlandı")

    kaydet(lime_mat, f"lime_cicids_{model_adi}.pkl")
    lime_degerleri[model_adi] = lime_mat
    print(f"  ✅ {model_adi} LIME tamamlandı")

print("\n🎉 SHAP + LIME hesaplama tamamlandı!")
print(f"  SHAP: {list(shap_degerleri.keys())}")
print(f"  LIME: {list(lime_degerleri.keys())}")

In [ ]:
import pandas as pd
import numpy as np
import os, pickle, psutil, warnings
warnings.filterwarnings('ignore')

CHECKPOINT_DIR = r"str(CICIDS2017_DIR)"

def yukle(dosya_adi):
    yol = os.path.join(CHECKPOINT_DIR, dosya_adi)
    with open(yol, 'rb') as f:
        return pickle.load(f)

def kaydet(nesne, dosya_adi):
    yol = os.path.join(CHECKPOINT_DIR, dosya_adi)
    with open(yol, 'wb') as f:
        pickle.dump(nesne, f)
    print(f"  💾 Kaydedildi: {dosya_adi}")

def kontrol_et(dosya_adi):
    return os.path.exists(os.path.join(CHECKPOINT_DIR, dosya_adi))

def ram_goster(etiket=""):
    ram = psutil.virtual_memory()
    print(f"  🧠 RAM [{etiket}]: {ram.used/1e9:.1f} GB / {ram.total/1e9:.1f} GB")

# Önceki checkpoint'lerden gerekli değişkenleri yükle
veri           = yukle("veri_cicids.pkl")
X_scaled       = veri['X_scaled']
y              = veri['y']
sinif_isimleri = veri['sinif_isimleri']
feature_names  = veri['feature_names']

# Seed 0 referans modeller
seed0_sonuclar = yukle("modeller_cicids_seed0.pkl")
ref_modeller   = {}
ref_X_test     = None
ref_y_test     = None
for kayit in seed0_sonuclar:
    if kayit['fold'] == 0:
        ref_modeller[kayit['model_adi']] = kayit['model_obj']
        if ref_X_test is None:
            ref_X_test = kayit['X_test']
            ref_y_test = kayit['y_test']

# 500 örnek
from sklearn.model_selection import StratifiedShuffleSplit
sss = StratifiedShuffleSplit(n_splits=1, test_size=500, random_state=42)
_, ornek_idx = next(sss.split(ref_X_test, ref_y_test))
X_ornek = ref_X_test[ornek_idx]
y_ornek = ref_y_test[ornek_idx]

# SHAP + LIME değerleri
shap_degerleri = {m: yukle(f"shap_cicids_{m}.pkl") for m in ref_modeller}
lime_degerleri = {m: yukle(f"lime_cicids_{m}.pkl") for m in ref_modeller}

print("✅ Her şey yüklendi, RQ1 kodunu çalıştırabilirsin.")
print(f"   X_ornek: {X_ornek.shape}, modeller: {list(ref_modeller.keys())}")

In [ ]:
# ============================================================
# CIC-IDS-2017 — RQ1: Pertürbasyon Kararlılığı
# %1, %3, %5 Gaussian gürültü, 500 örnek × 10 tekrar
# ============================================================

from scipy.stats import spearmanr
import numpy as np
import warnings
warnings.filterwarnings('ignore')

GURULTU_ORANLAR = [0.01, 0.03, 0.05]
N_TEKRAR        = 10
MODEL_SIRASI    = ['RandomForest', 'ExtraTrees', 'DecisionTree',
                   'XGBoost', 'LightGBM', 'CatBoost', 'LogisticRegression']

def shap_onem_vektoru(shap_abs):
    """(500,78,12) veya (500,78) → (500,78) ortalama önem"""
    if shap_abs.ndim == 3:
        return shap_abs.mean(axis=2)
    return shap_abs

def hesapla_shap_perturb(model, X, shap_abs_orig, gurultu_oran, n_tekrar, random_state=42):
    """Gürültülü SHAP Spearman r değerleri"""
    np.random.seed(random_state)
    orig_vec = shap_onem_vektoru(shap_abs_orig)  # (500,78)
    spearman_listesi = []

    for tekrar in range(n_tekrar):
        gurultu = np.random.normal(0, gurultu_oran * np.std(X, axis=0), X.shape)
        X_gurultulu = X + gurultu

        try:
            if hasattr(model, 'estimators_') or hasattr(model, 'tree_'):
                import shap
                explainer  = shap.TreeExplainer(model)
                shap_gurlt = explainer.shap_values(X_gurultulu)
            elif str(type(model).__name__) == 'LogisticRegression':
                import shap
                explainer  = shap.LinearExplainer(model, X)
                shap_gurlt = explainer.shap_values(X_gurultulu)
            else:
                import shap
                explainer  = shap.TreeExplainer(model)
                shap_gurlt = explainer.shap_values(X_gurultulu)

            gurlt_vec = shap_onem_vektoru(
                np.array(shap_gurlt) if isinstance(shap_gurlt, list)
                else shap_gurlt
            )
            if gurlt_vec.ndim == 3:
                gurlt_vec = gurlt_vec.mean(axis=2) if gurlt_vec.shape[0] != X.shape[0] else gurlt_vec.mean(axis=2)
            # Her örnek için Spearman r
            for i in range(len(X)):
                r, _ = spearmanr(orig_vec[i], gurlt_vec[i])
                if not np.isnan(r):
                    spearman_listesi.append(r)
        except Exception as e:
            print(f"      Hata (tekrar {tekrar}): {e}")

    return spearman_listesi

def hesapla_lime_perturb(model, X, lime_orig, gurultu_oran, n_tekrar,
                          feature_names, sinif_isimleri, random_state=42):
    """Gürültülü LIME Spearman r değerleri — sadece 50 örnek (yavaş)"""
    import lime.lime_tabular
    np.random.seed(random_state)
    N_LIME = 50  # LIME yavaş, 50 örnek yeterli
    spearman_listesi = []

    lime_exp = lime.lime_tabular.LimeTabularExplainer(
        training_data=X, feature_names=feature_names,
        class_names=sinif_isimleri, mode='classification',
        random_state=random_state, discretize_continuous=True
    )

    for tekrar in range(n_tekrar):
        gurultu  = np.random.normal(0, gurultu_oran * np.std(X, axis=0), X.shape)
        X_gurult = X + gurultu

        for i in range(N_LIME):
            try:
                exp  = lime_exp.explain_instance(
                    X_gurult[i], model.predict_proba,
                    num_features=len(feature_names), num_samples=500, top_labels=1
                )
                label    = exp.available_labels()[0]
                lime_map = dict(exp.as_list(label=label))
                gurlt_vec = np.zeros(len(feature_names))
                for j, feat in enumerate(feature_names):
                    for key, val in lime_map.items():
                        if feat in key:
                            gurlt_vec[j] = abs(val)
                            break
                r, _ = spearmanr(lime_orig[i], gurlt_vec)
                if not np.isnan(r):
                    spearman_listesi.append(r)
            except:
                pass

    return spearman_listesi

# ── Ana hesaplama ─────────────────────────────────────────────────────────
if kontrol_et("tum_perturb_sonuclar_cicids.pkl"):
    print("♻️  Pertürbasyon sonuçları checkpoint'ten yükleniyor...")
    tum_perturb = yukle("tum_perturb_sonuclar_cicids.pkl")
else:
    print("🚀 RQ1 Pertürbasyon analizi başlıyor...\n")
    tum_perturb = {}

    for model_adi in MODEL_SIRASI:
        print(f"\n{'='*50}")
        print(f"  Model: {model_adi}")
        print(f"{'='*50}")

        model     = ref_modeller[model_adi]
        shap_abs  = shap_degerleri[model_adi]
        lime_mat  = lime_degerleri[model_adi]
        tum_perturb[model_adi] = {}

        for oran in GURULTU_ORANLAR:
            etiket = f"pct{int(oran*100)}"
            print(f"\n  📊 Gürültü %{int(oran*100)}:")
            tum_perturb[model_adi][etiket] = {}

            # SHAP
            print(f"    🔄 SHAP pertürbasyon...")
            shap_r = hesapla_shap_perturb(
                model, X_ornek, shap_abs, oran, N_TEKRAR
            )
            tum_perturb[model_adi][etiket]['shap'] = {'spearman': shap_r}
            print(f"    ✅ SHAP Spearman r = {np.mean(shap_r):.4f} ± {np.std(shap_r):.4f} (n={len(shap_r)})")

            # LIME
            print(f"    🔄 LIME pertürbasyon (yavaş)...")
            lime_r = hesapla_lime_perturb(
                model, X_ornek, lime_mat, oran, N_TEKRAR,
                feature_names, sinif_isimleri
            )
            tum_perturb[model_adi][etiket]['lime'] = {'spearman': lime_r}
            print(f"    ✅ LIME Spearman r = {np.mean(lime_r):.4f} ± {np.std(lime_r):.4f} (n={len(lime_r)})")

        kaydet(tum_perturb, "tum_perturb_sonuclar_cicids.pkl")

# ── Özet tablo ────────────────────────────────────────────────────────────
print("\n\n📊 RQ1 — Pertürbasyon Kararlılığı Özet:")
print(f"\n{'Model':<22} {'SHAP %1':>8} {'SHAP %3':>8} {'SHAP %5':>8} | {'LIME %1':>8} {'LIME %3':>8} {'LIME %5':>8}")
print("-" * 80)

for model_adi in MODEL_SIRASI:
    shap_vals = []
    lime_vals = []
    for oran in ['pct1', 'pct3', 'pct5']:
        shap_vals.append(np.mean(tum_perturb[model_adi][oran]['shap']['spearman']))
        lime_vals.append(np.mean(tum_perturb[model_adi][oran]['lime']['spearman']))
    print(f"  {model_adi:<20} {shap_vals[0]:>8.4f} {shap_vals[1]:>8.4f} {shap_vals[2]:>8.4f} | "
          f"{lime_vals[0]:>8.4f} {lime_vals[1]:>8.4f} {lime_vals[2]:>8.4f}")

kaydet(tum_perturb, "tum_perturb_sonuclar_cicids.pkl")
print("\n✅ RQ1 tamamlandı!")

In [ ]:
import os, pickle, gc, time, numpy as np, pandas as pd
import shap, lime, lime.lime_tabular
from scipy.stats import spearmanr, pearsonr, mannwhitneyu
import warnings
warnings.filterwarnings('ignore')

CHECKPOINT_DIR = r"str(CICIDS2017_DIR)"

def kaydet(nesne, dosya_adi):
    yol = os.path.join(CHECKPOINT_DIR, dosya_adi)
    with open(yol, 'wb') as f:
        pickle.dump(nesne, f)
    boyut = os.path.getsize(yol) / 1024**2
    print(f"  💾 Kaydedildi: {dosya_adi} ({boyut:.1f} MB)")

def yukle(dosya_adi):
    yol = os.path.join(CHECKPOINT_DIR, dosya_adi)
    with open(yol, 'rb') as f:
        return pickle.load(f)

def kontrol_et(dosya_adi):
    return os.path.exists(os.path.join(CHECKPOINT_DIR, dosya_adi))

def bellek_temizle():
    gc.collect()

def ram_goster(mesaj=""):
    import psutil
    ram = psutil.virtual_memory()
    print(f"  🧠 RAM [{mesaj}]: Kullanılan={ram.used/1024**3:.1f}GB / "
          f"Toplam={ram.total/1024**3:.1f}GB (%{ram.percent})")

def jaccard_top_k(v1, v2, k):
    t1 = set(np.argsort(np.abs(v1))[-k:])
    t2 = set(np.argsort(np.abs(v2))[-k:])
    return len(t1 & t2) / len(t1 | t2) if len(t1 | t2) > 0 else 0.0

def lime_to_vec(lime_dict, feature_names):
    vec = np.zeros(len(feature_names))
    for fname, fval in lime_dict.items():
        clean = fname.split('<=')[0].split('>')[0].split('<')[0].strip()
        for i, fn in enumerate(feature_names):
            if fn == clean:
                vec[i] = fval
                break
    return vec

MODEL_SIRASI = ['RandomForest','ExtraTrees','DecisionTree',
                'XGBoost','LightGBM','CatBoost','LogisticRegression']

# ── Yükle ────────────────────────────────────────────────────────────────
print("📦 Yükleniyor...")
veri           = yukle("veri_cicids.pkl")
X_scaled       = veri['X_scaled']
y              = veri['y']
sinif_isimleri = veri['sinif_isimleri']
feature_names  = veri['feature_names']
print(f"✅ Veri: {X_scaled.shape}, {len(sinif_isimleri)} sınıf")

df_sonuc = yukle("df_sonuc_cicids.pkl")
print(f"✅ Performans tablosu: {df_sonuc.shape}")

seed0_data = yukle("modeller_cicids_seed0.pkl")
ref_X_test = seed0_data[0]['X_test']

np.random.seed(42)
ornek_idx    = np.random.choice(len(ref_X_test), size=500, replace=False)
X_orneklem   = ref_X_test[ornek_idx]
y_orneklem   = seed0_data[0]['y_test'][ornek_idx]

ref_modeller = {}
for s in seed0_data:
    if s['fold'] == 0:
        ref_modeller[s['model_adi']] = s['model_obj']
del seed0_data
bellek_temizle()
print(f"✅ Referans modeller: {list(ref_modeller.keys())}")

lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data = X_orneklem,
    feature_names = feature_names,
    class_names   = sinif_isimleri,
    mode          = 'classification',
    random_state  = 42
)
print(f"✅ LIME hazır")
print(f"\n🎉 Her şey yüklendi! Devam ediyoruz...")

In [ ]:
def get_shap_values_safe(model_adi, model, X):
    if model_adi in ['RandomForest','ExtraTrees','DecisionTree',
                     'XGBoost','LightGBM','CatBoost']:
        explainer = shap.TreeExplainer(model)
    else:
        explainer = shap.LinearExplainer(model, X)

    vals = explainer.shap_values(X)
    if isinstance(vals, list):
        vals = np.mean(np.abs(np.array(vals)), axis=0)
    elif vals.ndim == 3:
        vals = np.mean(np.abs(vals), axis=2)
    return explainer, vals

xai_sonuclar = {}

for model_adi in MODEL_SIRASI:
    dosya = f"xai_cicids_{model_adi}.pkl"
    if kontrol_et(dosya):
        print(f"♻️  {model_adi} XAI yüklendi, atlanıyor...")
        xai_sonuclar[model_adi] = yukle(dosya)
        continue

    print(f"\n{'='*55}")
    print(f"  🔷 {model_adi}")
    print(f"{'='*55}")
    ram_goster("başlangıç")

    model = ref_modeller[model_adi]

    # SHAP
    t0 = time.time()
    explainer, shap_vals = get_shap_values_safe(model_adi, model, X_orneklem)
    shap_sure = (time.time() - t0) / 500 * 1000
    print(f"  ✅ SHAP → shape={shap_vals.shape}, süre={shap_sure:.3f} ms/örnek")

    # LIME (500 örnek — Edge-IIoT ile aynı)
    lime_vals = []
    t0 = time.time()
    for i in range(500):
        exp = lime_explainer.explain_instance(
            X_orneklem[i],
            model.predict_proba,
            num_features=len(feature_names),
            num_samples=1000
        )
        lime_vals.append(lime_to_vec(dict(exp.as_list()), feature_names))
        if (i+1) % 100 == 0:
            print(f"    LIME {i+1}/500...")
    lime_sure = (time.time() - t0) / 500 * 1000
    print(f"  ✅ LIME → süre={lime_sure:.2f} ms/örnek")

    xai_sonuclar[model_adi] = {
        'shap_values' : shap_vals,
        'shap_sure'   : shap_sure,
        'lime_values' : np.array(lime_vals),
        'lime_sure'   : lime_sure,
    }
    kaydet(xai_sonuclar[model_adi], dosya)
moMODEL
    del explainer, shap_vals, lime_vals
    bellek_temizle()
    ram_goster("sonrası")

print(f"\n⏱️  Açıklama Süreleri (ms/örnek):")
print(f"{'Model':<20} {'SHAP':>10} {'LIME':>10} {'Oran':>10}")
print("-" * 54)
for m in MODEL_SIRASI:
    s = xai_sonuclar[m]['shap_sure']
    l = xai_sonuclar[m]['lime_sure']
    print(f"  {m:<18} {s:>10.3f} {l:>10.2f} {l/s:>9.1f}x")

print("\n✅ XAI analizi tamamlandı!")

In [ ]:
GURULTU_ORANLARI = [0.01, 0.03, 0.05]
N_PERTURBATION   = 10

print(f"🔀 Pertürbasyon analizi başlıyor: 7 model × 3 oran × 500 örnek × 10 tekrar\n")

tum_perturb_sonuclar = {}

for model_adi in MODEL_SIRASI:
    dosya = f"perturb_cicids_{model_adi}.pkl"
    if kontrol_et(dosya):
        print(f"♻️  {model_adi} pertürbasyon yüklendi, atlanıyor...")
        tum_perturb_sonuclar[model_adi] = yukle(dosya)
        continue

    print(f"\n{'='*55}")
    print(f"  🔷 {model_adi}")
    print(f"{'='*55}")
    ram_goster("başlangıç")

    model      = ref_modeller[model_adi]
    shap_orig  = xai_sonuclar[model_adi]['shap_values']   # (500, 78)
    lime_orig  = xai_sonuclar[model_adi]['lime_values']   # (500, 78)

    if model_adi in ['RandomForest','ExtraTrees','DecisionTree',
                     'XGBoost','LightGBM','CatBoost']:
        explainer = shap.TreeExplainer(model)
    else:
        explainer = shap.LinearExplainer(model, X_orneklem)

    model_sonuclar = {}

    for oran in GURULTU_ORANLARI:
        oran_key = f"pct{int(oran*100)}"
        sonuc = {
            'shap': {'spearman':[], 'jaccard_5':[], 'jaccard_10':[]},
            'lime': {'spearman':[], 'jaccard_5':[], 'jaccard_10':[]},
        }

        for ornek_i in range(500):
            x_orig = X_orneklem[ornek_i]
            so     = shap_orig[ornek_i]
            lo     = lime_orig[ornek_i]

            shap_sp_list=[] ; shap_j5_list=[] ; shap_j10_list=[]
            lime_sp_list=[] ; lime_j5_list=[] ; lime_j10_list=[]

            for _ in range(N_PERTURBATION):
                x_pert = x_orig.copy()
                x_pert += np.random.normal(0, oran, len(x_pert))

                # Pertürbe SHAP
                sv = explainer.shap_values(x_pert.reshape(1,-1))
                if isinstance(sv, list):
                    sv = np.mean(np.abs(np.array(sv)), axis=0)[0]
                elif sv.ndim == 3:
                    sv = np.mean(np.abs(sv), axis=2)[0]
                else:
                    sv = sv[0]

                # Pertürbe LIME
                exp = lime_explainer.explain_instance(
                    x_pert, model.predict_proba,
                    num_features=len(feature_names), num_samples=1000)
                lv = lime_to_vec(dict(exp.as_list()), feature_names)

                shap_sp_list.append(spearmanr(so, sv)[0])
                shap_j5_list.append(jaccard_top_k(so, sv, 5))
                shap_j10_list.append(jaccard_top_k(so, sv, 10))
                lime_sp_list.append(spearmanr(lo, lv)[0])
                lime_j5_list.append(jaccard_top_k(lo, lv, 5))
                lime_j10_list.append(jaccard_top_k(lo, lv, 10))

            sonuc['shap']['spearman'].append(np.nanmean(shap_sp_list))
            sonuc['shap']['jaccard_5'].append(np.nanmean(shap_j5_list))
            sonuc['shap']['jaccard_10'].append(np.nanmean(shap_j10_list))
            sonuc['lime']['spearman'].append(np.nanmean(lime_sp_list))
            sonuc['lime']['jaccard_5'].append(np.nanmean(lime_j5_list))
            sonuc['lime']['jaccard_10'].append(np.nanmean(lime_j10_list))

            if (ornek_i+1) % 100 == 0:
                print(f"    %{int(oran*100)} → {ornek_i+1}/500 tamamlandı")

        model_sonuclar[oran_key] = sonuc
        sp_shap = np.nanmean(sonuc['shap']['spearman'])
        sp_lime = np.nanmean(sonuc['lime']['spearman'])
        print(f"  ✅ %{int(oran*100)}: SHAP Spearman={sp_shap:.4f}, LIME Spearman={sp_lime:.4f}")

    tum_perturb_sonuclar[model_adi] = model_sonuclar
    kaydet(model_sonuclar, dosya)
    del explainer
    bellek_temizle()
    ram_goster("sonrası")

# Özet tablo
print(f"\n📊 Pertürbasyon Analizi Özeti (Spearman):")
print(f"{'Model':<20} {'%1 SHAP':>10} {'%1 LIME':>10} {'%3 SHAP':>10} {'%3 LIME':>10} {'%5 SHAP':>10} {'%5 LIME':>10}")
print("-" * 82)
for m in MODEL_SIRASI:
    s   = tum_perturb_sonuclar[m]
    row = f"  {m:<18}"
    for oran_key in ['pct1','pct3','pct5']:
        row += f" {np.nanmean(s[oran_key]['shap']['spearman']):>10.4f}"
        row += f" {np.nanmean(s[oran_key]['lime']['spearman']):>10.4f}"
    print(row)

kaydet(tum_perturb_sonuclar, "tum_perturb_sonuclar_cicids.pkl")
print("\n✅ Pertürbasyon analizi tamamlandı!")

In [ ]:
import os, pickle, gc, time, numpy as np, pandas as pd
import shap, lime, lime.lime_tabular
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')

CHECKPOINT_DIR = r"str(CICIDS2017_DIR)"

def kaydet(nesne, dosya_adi):
    yol = os.path.join(CHECKPOINT_DIR, dosya_adi)
    with open(yol, 'wb') as f:
        pickle.dump(nesne, f)
    boyut = os.path.getsize(yol) / 1024**2
    print(f"  💾 Kaydedildi: {dosya_adi} ({boyut:.1f} MB)")

def yukle(dosya_adi):
    yol = os.path.join(CHECKPOINT_DIR, dosya_adi)
    with open(yol, 'rb') as f:
        return pickle.load(f)

def kontrol_et(dosya_adi):
    return os.path.exists(os.path.join(CHECKPOINT_DIR, dosya_adi))

def bellek_temizle():
    gc.collect()

def ram_goster(mesaj=""):
    import psutil
    ram = psutil.virtual_memory()
    print(f"  🧠 RAM [{mesaj}]: Kullanılan={ram.used/1024**3:.1f}GB / "
          f"Toplam={ram.total/1024**3:.1f}GB (%{ram.percent})")

def jaccard_top_k(v1, v2, k):
    t1 = set(np.argsort(np.abs(v1))[-k:])
    t2 = set(np.argsort(np.abs(v2))[-k:])
    return len(t1 & t2) / len(t1 | t2) if len(t1 | t2) > 0 else 0.0

def lime_to_vec(lime_dict, feature_names):
    vec = np.zeros(len(feature_names))
    for fname, fval in lime_dict.items():
        clean = fname.split('<=')[0].split('>')[0].split('<')[0].strip()
        for i, fn in enumerate(feature_names):
            if fn == clean:
                vec[i] = fval
                break
    return vec

MODEL_SIRASI = ['RandomForest','ExtraTrees','DecisionTree',
                'XGBoost','LightGBM','CatBoost','LogisticRegression']

# ── Veri ─────────────────────────────────────────────────────────────────
print("📦 Yükleniyor...")
veri           = yukle("veri_cicids.pkl")
X_scaled       = veri['X_scaled']
y              = veri['y']
sinif_isimleri = veri['sinif_isimleri']
feature_names  = veri['feature_names']
print(f"✅ Veri: {X_scaled.shape}, {len(sinif_isimleri)} sınıf")

df_sonuc = yukle("df_sonuc_cicids.pkl")
print(f"✅ Performans tablosu: {df_sonuc.shape}")

# ── Referans modeller (seed=0, fold=0) ───────────────────────────────────
seed0_data = yukle("modeller_cicids_seed0.pkl")
ref_X_test = seed0_data[0]['X_test']
np.random.seed(42)
ornek_idx  = np.random.choice(len(ref_X_test), size=500, replace=False)
X_orneklem = ref_X_test[ornek_idx]
y_orneklem = seed0_data[0]['y_test'][ornek_idx]

ref_modeller = {}
for s in seed0_data:
    if s['fold'] == 0:
        ref_modeller[s['model_adi']] = s['model_obj']
del seed0_data
bellek_temizle()
print(f"✅ Referans modeller: {list(ref_modeller.keys())}")

# ── XAI sonuçlarını yükle (varsa) ────────────────────────────────────────
xai_sonuclar = {}
for m in MODEL_SIRASI:
    dosya = f"xai_cicids_{m}.pkl"
    if kontrol_et(dosya):
        xai_sonuclar[m] = yukle(dosya)
print(f"✅ XAI sonuçları: {list(xai_sonuclar.keys())}")

# ── Pertürbasyon sonuçlarını yükle (varsa) ────────────────────────────────
tum_perturb_sonuclar = {}
for m in MODEL_SIRASI:
    dosya = f"perturb_cicids_{m}.pkl"
    if kontrol_et(dosya):
        tum_perturb_sonuclar[m] = yukle(dosya)
print(f"✅ Pertürbasyon sonuçları: {list(tum_perturb_sonuclar.keys())}")

# ── LIME açıklayıcı ───────────────────────────────────────────────────────
lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data = X_orneklem,
    feature_names = feature_names,
    class_names   = sinif_isimleri,
    mode          = 'classification',
    random_state  = 42
)
print(f"✅ LIME hazır")
print(f"\n🎉 Her şey yüklendi!")

In [ ]:
def get_shap_values_safe(model_adi, model, X):
    if model_adi in ['RandomForest','ExtraTrees','DecisionTree',
                     'XGBoost','LightGBM','CatBoost']:
        explainer = shap.TreeExplainer(model)
    else:
        explainer = shap.LinearExplainer(model, X)
    vals = explainer.shap_values(X)
    if isinstance(vals, list):
        vals = np.mean(np.abs(np.array(vals)), axis=0)
    elif vals.ndim == 3:
        vals = np.mean(np.abs(vals), axis=2)
    return explainer, vals

for model_adi in MODEL_SIRASI:
    dosya = f"xai_cicids_{model_adi}.pkl"
    if kontrol_et(dosya):
        print(f"♻️  {model_adi} XAI yüklendi, atlanıyor...")
        if model_adi not in xai_sonuclar:
            xai_sonuclar[model_adi] = yukle(dosya)
        continue

    print(f"\n{'='*55}")
    print(f"  🔷 {model_adi}")
    print(f"{'='*55}")
    ram_goster("başlangıç")

    model = ref_modeller[model_adi]

    t0 = time.time()
    explainer, shap_vals = get_shap_values_safe(model_adi, model, X_orneklem)
    shap_sure = (time.time() - t0) / 500 * 1000
    print(f"  ✅ SHAP → shape={shap_vals.shape}, süre={shap_sure:.3f} ms/örnek")

    lime_vals = []
    t0 = time.time()
    for i in range(500):
        exp = lime_explainer.explain_instance(
            X_orneklem[i], model.predict_proba,
            num_features=len(feature_names), num_samples=1000)
        lime_vals.append(lime_to_vec(dict(exp.as_list()), feature_names))
        if (i+1) % 100 == 0:
            print(f"    LIME {i+1}/500...")
    lime_sure = (time.time() - t0) / 500 * 1000
    print(f"  ✅ LIME → süre={lime_sure:.2f} ms/örnek")

    xai_sonuclar[model_adi] = {
        'shap_values': shap_vals,
        'shap_sure'  : shap_sure,
        'lime_values': np.array(lime_vals),
        'lime_sure'  : lime_sure,
    }
    kaydet(xai_sonuclar[model_adi], dosya)
    del explainer, shap_vals, lime_vals
    bellek_temizle()
    ram_goster("sonrası")

print(f"\n⏱️  Açıklama Süreleri (ms/örnek):")
print(f"{'Model':<20} {'SHAP':>10} {'LIME':>10} {'Oran':>10}")
print("-" * 54)
for m in MODEL_SIRASI:
    s = xai_sonuclar[m]['shap_sure']
    l = xai_sonuclar[m]['lime_sure']
    print(f"  {m:<18} {s:>10.3f} {l:>10.2f} {l/s:>9.1f}x")
print("\n✅ XAI analizi tamamlandı!")

In [ ]:
GURULTU_ORANLARI = [0.01, 0.03, 0.05]
N_PERTURBATION   = 10

def get_sv_single(model_adi, model, explainer, x_pert):
    if model_adi == 'CatBoost':
        from catboost import Pool
        pool = Pool(x_pert.reshape(1, -1))
        sv = model.get_feature_importance(pool, type='ShapValues')
        sv = np.array(sv)
        if sv.ndim == 3:
            sv = np.mean(np.abs(sv[0, :, :-1]), axis=0)
        elif sv.ndim == 2 and sv.shape[0] == 1:
            sv = np.abs(sv[0, :-1])
        elif sv.ndim == 2:
            sv = np.mean(np.abs(sv[:, :-1]), axis=0)
        return sv
    else:
        sv = explainer.shap_values(x_pert.reshape(1, -1))
        if isinstance(sv, list):
            sv = np.mean(np.abs(np.array(sv)), axis=0)[0]
        elif sv.ndim == 3:
            sv = np.mean(np.abs(sv), axis=2)[0]
        else:
            sv = sv[0]
        return sv

print(f"🔀 Pertürbasyon analizi başlıyor: 7 model × 3 oran × 500 örnek × 10 tekrar\n")

tum_perturb_sonuclar = {}

for model_adi in MODEL_SIRASI:
    dosya = f"perturb_cicids_{model_adi}.pkl"
    if kontrol_et(dosya):
        print(f"♻️  {model_adi} pertürbasyon yüklendi, atlanıyor...")
        tum_perturb_sonuclar[model_adi] = yukle(dosya)
        continue

    print(f"\n{'='*55}")
    print(f"  🔷 {model_adi}")
    print(f"{'='*55}")
    ram_goster("başlangıç")

    model     = ref_modeller[model_adi]
    shap_orig = xai_sonuclar[model_adi]['shap_values']
    lime_orig = xai_sonuclar[model_adi]['lime_values']

    if model_adi == 'CatBoost':
        explainer = None
    elif model_adi == 'LogisticRegression':
        explainer = shap.LinearExplainer(model, X_orneklem)
    else:
        explainer = shap.TreeExplainer(model)

    model_sonuclar = {}

    for oran in GURULTU_ORANLARI:
        oran_key = f"pct{int(oran*100)}"
        sonuc = {
            'shap': {'spearman':[], 'jaccard_5':[], 'jaccard_10':[]},
            'lime': {'spearman':[], 'jaccard_5':[], 'jaccard_10':[]},
        }

        for ornek_i in range(500):
            x_orig = X_orneklem[ornek_i]
            so     = shap_orig[ornek_i]
            lo     = lime_orig[ornek_i]

            shap_sp_list=[] ; shap_j5_list=[] ; shap_j10_list=[]
            lime_sp_list=[] ; lime_j5_list=[] ; lime_j10_list=[]

            for _ in range(N_PERTURBATION):
                x_pert = x_orig.copy()
                x_pert += np.random.normal(0, oran, len(x_pert))

                sv = get_sv_single(model_adi, model, explainer, x_pert)

                # sv boyutu so ile eşleşmiyorsa atla
                if sv.shape != so.shape:
                    continue

                exp_lime = lime_explainer.explain_instance(
                    x_pert, model.predict_proba,
                    num_features=len(feature_names), num_samples=1000)
                lv = lime_to_vec(dict(exp_lime.as_list()), feature_names)

                shap_sp_list.append(spearmanr(so, sv)[0])
                shap_j5_list.append(jaccard_top_k(so, sv, 5))
                shap_j10_list.append(jaccard_top_k(so, sv, 10))
                lime_sp_list.append(spearmanr(lo, lv)[0])
                lime_j5_list.append(jaccard_top_k(lo, lv, 5))
                lime_j10_list.append(jaccard_top_k(lo, lv, 10))

            sonuc['shap']['spearman'].append(np.nanmean(shap_sp_list))
            sonuc['shap']['jaccard_5'].append(np.nanmean(shap_j5_list))
            sonuc['shap']['jaccard_10'].append(np.nanmean(shap_j10_list))
            sonuc['lime']['spearman'].append(np.nanmean(lime_sp_list))
            sonuc['lime']['jaccard_5'].append(np.nanmean(lime_j5_list))
            sonuc['lime']['jaccard_10'].append(np.nanmean(lime_j10_list))

            if (ornek_i+1) % 100 == 0:
                print(f"    %{int(oran*100)} → {ornek_i+1}/500 tamamlandı")

        model_sonuclar[oran_key] = sonuc
        sp_shap = np.nanmean(sonuc['shap']['spearman'])
        sp_lime = np.nanmean(sonuc['lime']['spearman'])
        print(f"  ✅ %{int(oran*100)}: SHAP Spearman={sp_shap:.4f}, LIME Spearman={sp_lime:.4f}")

    tum_perturb_sonuclar[model_adi] = model_sonuclar
    kaydet(model_sonuclar, dosya)
    if explainer is not None:
        del explainer
    bellek_temizle()
    ram_goster("sonrası")

# Özet tablo
print(f"\n📊 Pertürbasyon Analizi Özeti (Spearman):")
print(f"{'Model':<20} {'%1 SHAP':>10} {'%1 LIME':>10} {'%3 SHAP':>10} {'%3 LIME':>10} {'%5 SHAP':>10} {'%5 LIME':>10}")
print("-" * 82)
for m in MODEL_SIRASI:
    s   = tum_perturb_sonuclar[m]
    row = f"  {m:<18}"
    for oran_key in ['pct1','pct3','pct5']:
        row += f" {np.nanmean(s[oran_key]['shap']['spearman']):>10.4f}"
        row += f" {np.nanmean(s[oran_key]['lime']['spearman']):>10.4f}"
    print(row)

kaydet(tum_perturb_sonuclar, "tum_perturb_sonuclar_cicids.pkl")
print("\n✅ Pertürbasyon analizi tamamlandı!")

In [ ]:
# ============================================================
# CICIDS2017 RQ2 - CatBoost kernel-guvenli devam hucresi
# Eski RQ2 hucresindeki CatBoost + shap.TreeExplainer yolu kernel patlatiyordu.
# Bu hucre CatBoost icin shap kutuphanesini hic kullanmaz; CatBoost native SHAP
# degerlerini tek ornek batch'leriyle hesaplar ve mevcut model checkpoint'lerini korur.
# ============================================================

import os, pickle, gc
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from catboost import Pool
import warnings
warnings.filterwarnings("ignore")

CHECKPOINT_DIR = r"str(CICIDS2017_DIR)"


def kaydet(nesne, dosya_adi):
    yol = os.path.join(CHECKPOINT_DIR, dosya_adi)
    with open(yol, "wb") as f:
        pickle.dump(nesne, f, protocol=pickle.HIGHEST_PROTOCOL)
    boyut = os.path.getsize(yol) / 1024**2
    print(f"  Kaydedildi: {dosya_adi} ({boyut:.1f} MB)")


def yukle(dosya_adi):
    yol = os.path.join(CHECKPOINT_DIR, dosya_adi)
    with open(yol, "rb") as f:
        return pickle.load(f)


def kontrol_et(dosya_adi):
    return os.path.exists(os.path.join(CHECKPOINT_DIR, dosya_adi))


def bellek_temizle():
    gc.collect()


def normalize_ref_shap(sv):
    arr = np.asarray(sv)
    if arr.ndim == 3:
        # Referans dosyasinda CatBoost genelde (ornek, ozellik, sinif).
        arr = np.mean(np.abs(arr), axis=2)
    elif isinstance(sv, list):
        arr = np.mean(np.abs(np.asarray(sv)), axis=0)
    return np.asarray(arr, dtype=np.float32)


def normalize_catboost_native_shap(raw_sv, n_features):
    arr = np.asarray(raw_sv)

    if arr.ndim == 2:
        # Binary/regression: (ornek, ozellik + expected_value)
        if arr.shape[1] == n_features + 1:
            arr = arr[:, :n_features]
        elif arr.shape[1] != n_features:
            raise ValueError(f"Beklenmeyen CatBoost SHAP shape: {arr.shape}")
        return np.abs(arr).astype(np.float32)

    if arr.ndim == 3:
        # Multiclass CatBoost surume gore iki bicimde donebilir:
        # (ornek, sinif, ozellik + expected_value) veya (ornek, ozellik + expected_value, sinif)
        if arr.shape[-1] == n_features + 1:
            arr = arr[..., :n_features]
            return np.mean(np.abs(arr), axis=1).astype(np.float32)
        if arr.shape[1] == n_features + 1:
            arr = arr[:, :n_features, :]
            return np.mean(np.abs(arr), axis=2).astype(np.float32)
        if arr.shape[-1] == n_features:
            return np.mean(np.abs(arr), axis=1).astype(np.float32)
        if arr.shape[1] == n_features:
            return np.mean(np.abs(arr), axis=2).astype(np.float32)

    raise ValueError(f"Beklenmeyen CatBoost SHAP shape: {arr.shape}")


def catboost_shap_satir_satir(model, X):
    X = np.asarray(X, dtype=np.float32)
    n_features = X.shape[1]
    parcalar = []

    for i in range(len(X)):
        raw_sv = model.get_feature_importance(
            Pool(X[i:i+1]),
            type="ShapValues",
            shap_calc_type="Approximate",
        )
        parcalar.append(normalize_catboost_native_shap(raw_sv, n_features))
        del raw_sv
        bellek_temizle()

    return np.vstack(parcalar).astype(np.float32)


def jaccard_top_k(v1, v2, k):
    v1 = np.asarray(v1).ravel()
    v2 = np.asarray(v2).ravel()
    t1 = set(np.argsort(np.abs(v1))[-k:])
    t2 = set(np.argsort(np.abs(v2))[-k:])
    return len(t1 & t2) / len(t1 | t2) if len(t1 | t2) > 0 else 0.0


MODEL_SIRASI = [
    "RandomForest", "ExtraTrees", "DecisionTree",
    "XGBoost", "LightGBM", "CatBoost", "LogisticRegression"
]

print("CatBoost referans SHAP yukleniyor...")
xai_cat = yukle("xai_cicids_CatBoost.pkl")
ref_sv = normalize_ref_shap(xai_cat["shap_values"][:50])
del xai_cat
bellek_temizle()
print(f"Referans hazir: {ref_sv.shape}")

catboost_dosya = "df_var_cicids_CatBoost.pkl"
if kontrol_et(catboost_dosya):
    print(f"{catboost_dosya} zaten var, CatBoost tekrar hesaplanmayacak.")
else:
    print("\nCatBoost RQ2 basliyor: 10 seed x 5 fold, satir satir native SHAP")
    model_kayitlar = []

    for seed in range(10):
        print(f"\n  seed={seed} yukleniyor...")
        seed_data = yukle(f"modeller_cicids_seed{seed}.pkl")

        cat_kayitlar = [k for k in seed_data if k["model_adi"] == "CatBoost"]
        del seed_data
        bellek_temizle()
        print(f"  CatBoost fold sayisi: {len(cat_kayitlar)}")

        for kayit in cat_kayitlar:
            fold = kayit["fold"]
            model_obj = kayit["model_obj"]
            X_sub = np.asarray(kayit["X_test"][:50], dtype=np.float32)
            f1 = kayit["f1_macro"]

            try:
                print(f"    fold={fold} SHAP hesaplaniyor...")
                sv = catboost_shap_satir_satir(model_obj, X_sub)

                sp_list, j5_list, j10_list = [], [], []
                for i in range(min(len(ref_sv), len(sv))):
                    if ref_sv[i].shape == sv[i].shape:
                        r, _ = spearmanr(ref_sv[i], sv[i])
                        if not np.isnan(r):
                            sp_list.append(r)
                        j5_list.append(jaccard_top_k(ref_sv[i], sv[i], 5))
                        j10_list.append(jaccard_top_k(ref_sv[i], sv[i], 10))

                model_kayitlar.append({
                    "model_adi": "CatBoost",
                    "seed": seed,
                    "fold": fold,
                    "f1_macro": f1,
                    "spearman": np.nanmean(sp_list),
                    "jaccard_5": np.nanmean(j5_list),
                    "jaccard_10": np.nanmean(j10_list),
                })

                print(
                    f"    tamam fold={fold}: sv={sv.shape}, "
                    f"spearman={np.nanmean(sp_list):.4f}, "
                    f"j@5={np.nanmean(j5_list):.4f}, j@10={np.nanmean(j10_list):.4f}"
                )
                del sv

            except Exception as e:
                print(f"    UYARI seed={seed} fold={fold}: {e}")

            del model_obj, X_sub
            bellek_temizle()

        del cat_kayitlar
        bellek_temizle()

    kaydet(model_kayitlar, catboost_dosya)
    print(f"\nCatBoost tamamlandi: {len(model_kayitlar)} kayit")

print("\nModel checkpoint'leri birlestiriliyor...")
tum_kayitlar = []
eksik = []
for model_adi in MODEL_SIRASI:
    dosya = f"df_var_cicids_{model_adi}.pkl"
    if kontrol_et(dosya):
        tum_kayitlar.extend(yukle(dosya))
    else:
        eksik.append(model_adi)

df_var = pd.DataFrame(tum_kayitlar)
kaydet(df_var, "df_var_cicids.pkl")

print("\nRQ2 - Model Varyasyonu Tutarliligi:")
print(f"\n{'Model':<22} {'N':>4} {'Spearman':>10} {'Jaccard@5':>10} {'Jaccard@10':>10}")
print("-" * 62)
for model_adi in MODEL_SIRASI:
    alt = df_var[df_var["model_adi"] == model_adi]
    if len(alt) == 0:
        print(f"  {model_adi:<20} {0:>4} {'eksik':>10} {'eksik':>10} {'eksik':>10}")
    else:
        print(
            f"  {model_adi:<20} {len(alt):>4} "
            f"{alt['spearman'].mean():>10.4f} "
            f"{alt['jaccard_5'].mean():>10.4f} "
            f"{alt['jaccard_10'].mean():>10.4f}"
        )

if eksik:
    print(f"\nEksik modeller: {eksik}")
else:
    print("\nRQ2 tamamlandi!")


In [ ]:
# ============================================================
# CICIDS2017 RQ2 - LogisticRegression eksigini tamamlama hucresi
# CatBoost tamamlandi; bu hucre yalniz eksik LR checkpoint'ini uretir
# ve tum model checkpoint'lerini df_var_cicids.pkl icinde birlestirir.
# ============================================================

import os, pickle, gc
import numpy as np
import pandas as pd
import shap
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings("ignore")

CHECKPOINT_DIR = r"str(CICIDS2017_DIR)"


def kaydet(nesne, dosya_adi):
    yol = os.path.join(CHECKPOINT_DIR, dosya_adi)
    with open(yol, "wb") as f:
        pickle.dump(nesne, f, protocol=pickle.HIGHEST_PROTOCOL)
    boyut = os.path.getsize(yol) / 1024**2
    print(f"  Kaydedildi: {dosya_adi} ({boyut:.1f} MB)")


def yukle(dosya_adi):
    yol = os.path.join(CHECKPOINT_DIR, dosya_adi)
    with open(yol, "rb") as f:
        return pickle.load(f)


def kontrol_et(dosya_adi):
    return os.path.exists(os.path.join(CHECKPOINT_DIR, dosya_adi))


def bellek_temizle():
    gc.collect()


def normalize_shap_values(sv):
    if isinstance(sv, list):
        return np.mean(np.abs(np.asarray(sv)), axis=0).astype(np.float32)

    arr = np.asarray(sv)
    if arr.ndim == 3:
        return np.mean(np.abs(arr), axis=2).astype(np.float32)
    return arr.astype(np.float32)


def jaccard_top_k(v1, v2, k):
    v1 = np.asarray(v1).ravel()
    v2 = np.asarray(v2).ravel()
    t1 = set(np.argsort(np.abs(v1))[-k:])
    t2 = set(np.argsort(np.abs(v2))[-k:])
    return len(t1 & t2) / len(t1 | t2) if len(t1 | t2) > 0 else 0.0


MODEL_SIRASI = [
    "RandomForest", "ExtraTrees", "DecisionTree",
    "XGBoost", "LightGBM", "CatBoost", "LogisticRegression"
]

lr_dosya = "df_var_cicids_LogisticRegression.pkl"

print("LogisticRegression referans SHAP yukleniyor...")
xai_lr = yukle("xai_cicids_LogisticRegression.pkl")
ref_sv = normalize_shap_values(xai_lr["shap_values"][:50]).copy()
del xai_lr
bellek_temizle()
print(f"Referans hazir: {ref_sv.shape}")

if kontrol_et(lr_dosya):
    print(f"{lr_dosya} zaten var, LR tekrar hesaplanmayacak.")
else:
    print("\nLogisticRegression RQ2 basliyor: 10 seed x 5 fold")
    lr_kayitlar = []

    for seed in range(10):
        print(f"\n  seed={seed} yukleniyor...")
        seed_data = yukle(f"modeller_cicids_seed{seed}.pkl")
        ilgili = [k for k in seed_data if k["model_adi"] == "LogisticRegression"]
        del seed_data
        bellek_temizle()
        print(f"  LR fold sayisi: {len(ilgili)}")

        for kayit in ilgili:
            fold = kayit["fold"]
            model_obj = kayit["model_obj"]
            X_sub = np.asarray(kayit["X_test"][:50], dtype=np.float32)
            f1 = kayit["f1_macro"]

            try:
                print(f"    fold={fold} LinearExplainer hesaplaniyor...")
                exp = shap.LinearExplainer(model_obj, X_sub)
                sv = normalize_shap_values(exp.shap_values(X_sub))
                del exp
                bellek_temizle()

                sp_list, j5_list, j10_list = [], [], []
                for i in range(min(len(ref_sv), len(sv))):
                    if ref_sv[i].shape == sv[i].shape:
                        r, _ = spearmanr(ref_sv[i], sv[i])
                        if not np.isnan(r):
                            sp_list.append(r)
                        j5_list.append(jaccard_top_k(ref_sv[i], sv[i], 5))
                        j10_list.append(jaccard_top_k(ref_sv[i], sv[i], 10))

                lr_kayitlar.append({
                    "model_adi": "LogisticRegression",
                    "seed": seed,
                    "fold": fold,
                    "f1_macro": f1,
                    "spearman": np.nanmean(sp_list),
                    "jaccard_5": np.nanmean(j5_list),
                    "jaccard_10": np.nanmean(j10_list),
                })

                print(
                    f"    tamam fold={fold}: sv={sv.shape}, "
                    f"spearman={np.nanmean(sp_list):.4f}, "
                    f"j@5={np.nanmean(j5_list):.4f}, j@10={np.nanmean(j10_list):.4f}"
                )
                del sv

            except Exception as e:
                print(f"    UYARI seed={seed} fold={fold}: {e}")

            del model_obj, X_sub
            bellek_temizle()

        del ilgili
        bellek_temizle()

    kaydet(lr_kayitlar, lr_dosya)
    print(f"\nLogisticRegression tamamlandi: {len(lr_kayitlar)} kayit")

print("\nModel checkpoint'leri birlestiriliyor...")
tum_kayitlar = []
eksik = []
for model_adi in MODEL_SIRASI:
    dosya = f"df_var_cicids_{model_adi}.pkl"
    if kontrol_et(dosya):
        tum_kayitlar.extend(yukle(dosya))
    else:
        eksik.append(model_adi)

df_var = pd.DataFrame(tum_kayitlar)
kaydet(df_var, "df_var_cicids.pkl")

print("\nRQ2 - Model Varyasyonu Tutarliligi:")
print(f"\n{'Model':<22} {'N':>4} {'Spearman':>10} {'Jaccard@5':>10} {'Jaccard@10':>10}")
print("-" * 62)
for model_adi in MODEL_SIRASI:
    alt = df_var[df_var["model_adi"] == model_adi]
    if len(alt) == 0:
        print(f"  {model_adi:<20} {0:>4} {'eksik':>10} {'eksik':>10} {'eksik':>10}")
    else:
        print(
            f"  {model_adi:<20} {len(alt):>4} "
            f"{alt['spearman'].mean():>10.4f} "
            f"{alt['jaccard_5'].mean():>10.4f} "
            f"{alt['jaccard_10'].mean():>10.4f}"
        )

if eksik:
    print(f"\nEksik modeller: {eksik}")
else:
    print("\nRQ2 tamamlandi!")


In [ ]:
import os, pickle, gc, numpy as np, pandas as pd
from scipy.stats import spearmanr, pearsonr, mannwhitneyu
from sklearn.metrics import f1_score, accuracy_score
import warnings
warnings.filterwarnings('ignore')

CHECKPOINT_DIR = r"str(CICIDS2017_DIR)"

def kaydet(nesne, dosya_adi):
    yol = os.path.join(CHECKPOINT_DIR, dosya_adi)
    with open(yol, 'wb') as f:
        pickle.dump(nesne, f)
    boyut = os.path.getsize(yol) / 1024**2
    print(f"  💾 Kaydedildi: {dosya_adi} ({boyut:.1f} MB)")

def yukle(dosya_adi):
    yol = os.path.join(CHECKPOINT_DIR, dosya_adi)
    with open(yol, 'rb') as f:
        return pickle.load(f)

def kontrol_et(dosya_adi):
    return os.path.exists(os.path.join(CHECKPOINT_DIR, dosya_adi))

MODEL_SIRASI = ['RandomForest','ExtraTrees','DecisionTree',
                'XGBoost','LightGBM','CatBoost','LogisticRegression']

# ============================================================
# RQ3 — F1 ile SHAP Kararlılığı İlişkisi
# ============================================================
print("📊 RQ3 — F1 ile SHAP Kararlılığı İlişkisi\n")

if kontrol_et("df_rq3_cicids.pkl"):
    print("♻️  RQ3 checkpoint'ten yükleniyor...")
    df_rq3 = yukle("df_rq3_cicids.pkl")
else:
    df_var   = yukle("df_var_cicids.pkl")
    df_rq3   = df_var[['model_adi','f1_macro','spearman']].copy()
    kaydet(df_rq3, "df_rq3_cicids.pkl")

r_genel, p_genel = pearsonr(df_rq3['f1_macro'], df_rq3['spearman'])
print(f"Genel Pearson r = {r_genel:.4f}, p = {p_genel:.4f}")

medyan_f1   = df_rq3['f1_macro'].median()
grup_yuksek = df_rq3[df_rq3['f1_macro'] >= medyan_f1]['spearman']
grup_dusuk  = df_rq3[df_rq3['f1_macro'] <  medyan_f1]['spearman']
mw_stat, mw_p = mannwhitneyu(grup_yuksek, grup_dusuk, alternative='two-sided')
print(f"Mann-Whitney U = {mw_stat:.1f}, p = {mw_p:.4f}")

print(f"\n{'Model':<22} {'Pearson r':>10} {'p-value':>10} {'Anlamlı':>10}")
print("-" * 56)
for m in MODEL_SIRASI:
    alt = df_rq3[df_rq3['model_adi'] == m]
    if len(alt) < 3:
        continue
    r, p = pearsonr(alt['f1_macro'], alt['spearman'])
    anlam = "✅" if p < 0.05 else "❌"
    print(f"  {m:<20} {r:>10.4f} {p:>10.4f} {anlam:>10}")

print("\n✅ RQ3 tamamlandı!")

# ============================================================
# FİDELİTY ANALİZİ — Kümülatif SHAP Maskeleme
# ============================================================
print("\n\n📊 Fidelity Analizi başlıyor...\n")

if kontrol_et("df_fidelity_cicids.pkl"):
    print("♻️  Fidelity checkpoint'ten yükleniyor...")
    df_fidelity = yukle("df_fidelity_cicids.pkl")
else:
    veri          = yukle("veri_cicids.pkl")
    feature_names = veri['feature_names']
    del veri
    gc.collect()

    seed0_data   = yukle("modeller_cicids_seed0.pkl")
    ref_X_test   = None
    ref_y_test   = None
    ref_modeller = {}
    for s in seed0_data:
        if s['fold'] == 0:
            ref_modeller[s['model_adi']] = s['model_obj']
            if ref_X_test is None:
                ref_X_test = s['X_test']
                ref_y_test = s['y_test']
    del seed0_data
    gc.collect()

    np.random.seed(42)
    fid_idx = np.random.choice(len(ref_X_test), size=200, replace=False)
    X_fid   = ref_X_test[fid_idx]
    y_fid   = ref_y_test[fid_idx]

    N_MASKELE_LIST    = list(range(1, 21))
    fidelity_kayitlar = []

    for model_adi in MODEL_SIRASI:
        print(f"  🔷 {model_adi}...")
        model    = ref_modeller[model_adi]
        xai      = yukle(f"xai_cicids_{model_adi}.pkl")
        shap_abs = xai['shap_values']   # (500, 78)
        del xai
        gc.collect()

        shap_200   = shap_abs[:200]
        global_imp = np.mean(np.abs(shap_200), axis=0)
        sirali_idx = np.argsort(global_imp)[::-1]

        y_pred_orig = model.predict(X_fid)
        f1_orig     = f1_score(y_fid, y_pred_orig, average='macro')
        acc_orig    = accuracy_score(y_fid, y_pred_orig)

        for n_mask in N_MASKELE_LIST:
            mask_idx = sirali_idx[:n_mask]
            X_masked = X_fid.copy()
            X_masked[:, mask_idx] = 0.0

            y_pred_mask = model.predict(X_masked)
            f1_mask     = f1_score(y_fid, y_pred_mask, average='macro')
            acc_mask    = accuracy_score(y_fid, y_pred_mask)

            fidelity_kayitlar.append({
                'model'    : model_adi,
                'xai'      : 'SHAP',
                'n_maskele': n_mask,
                'f1_orig'  : round(f1_orig, 4),
                'f1_mask'  : round(f1_mask, 4),
                'delta_f1' : round(f1_orig - f1_mask, 4),
                'acc_orig' : round(acc_orig, 4),
                'acc_mask' : round(acc_mask, 4),
            })

        del shap_abs
        gc.collect()

        d5 = next(r['delta_f1'] for r in fidelity_kayitlar
                  if r['model'] == model_adi and r['n_maskele'] == 5)
        print(f"    ΔF1@5 = {d5:.4f}")

    df_fidelity = pd.DataFrame(fidelity_kayitlar)
    kaydet(df_fidelity, "df_fidelity_cicids.pkl")

print(f"\n{'Model':<22} {'ΔF1@1':>8} {'ΔF1@5':>8} {'ΔF1@10':>8} {'ΔF1@20':>8}")
print("-" * 52)
for m in MODEL_SIRASI:
    alt = df_fidelity[df_fidelity['model'] == m]
    d1  = alt[alt['n_maskele']==1 ]['delta_f1'].values[0]
    d5  = alt[alt['n_maskele']==5 ]['delta_f1'].values[0]
    d10 = alt[alt['n_maskele']==10]['delta_f1'].values[0]
    d20 = alt[alt['n_maskele']==20]['delta_f1'].values[0]
    print(f"  {m:<20} {d1:>8.4f} {d5:>8.4f} {d10:>8.4f} {d20:>8.4f}")

print("\n✅ Fidelity tamamlandı!")

In [ ]:
import os, pickle, gc, numpy as np, pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon, pearsonr
from sklearn.metrics import f1_score
import warnings
warnings.filterwarnings('ignore')

CHECKPOINT_DIR = r"str(CICIDS2017_DIR)"
GORSEL_DIR     = r"str(CICIDS2017_DIR)"

def kaydet(nesne, dosya_adi):
    yol = os.path.join(CHECKPOINT_DIR, dosya_adi)
    with open(yol, 'wb') as f:
        pickle.dump(nesne, f)
    boyut = os.path.getsize(yol) / 1024**2
    print(f"  💾 Kaydedildi: {dosya_adi} ({boyut:.1f} MB)")

def yukle(dosya_adi):
    yol = os.path.join(CHECKPOINT_DIR, dosya_adi)
    with open(yol, 'rb') as f:
        return pickle.load(f)

def kontrol_et(dosya_adi):
    return os.path.exists(os.path.join(CHECKPOINT_DIR, dosya_adi))

MODEL_SIRASI = ['RandomForest','ExtraTrees','DecisionTree',
                'XGBoost','LightGBM','CatBoost','LogisticRegression']

RENKLER = {
    'RandomForest'      : '#2196F3',
    'ExtraTrees'        : '#4CAF50',
    'DecisionTree'      : '#FF9800',
    'XGBoost'           : '#F44336',
    'LightGBM'          : '#9C27B0',
    'CatBoost'          : '#00BCD4',
    'LogisticRegression': '#795548',
}
KISALTMA = {
    'RandomForest':'RF','ExtraTrees':'ET','DecisionTree':'DT',
    'XGBoost':'XGB','LightGBM':'LGBM','CatBoost':'CB',
    'LogisticRegression':'LR'
}

# ── Verileri yükle ────────────────────────────────────────────────────────
print("📦 Yükleniyor...")
tum_perturb = {m: yukle(f"perturb_cicids_{m}.pkl") for m in MODEL_SIRASI}
df_var      = yukle("df_var_cicids.pkl")
df_fidelity = yukle("df_fidelity_cicids.pkl")
df_sonuc    = yukle("df_sonuc_cicids.pkl")
print("✅ Hazır\n")

# ============================================================
# İSTATİSTİKSEL TESTLER — Wilcoxon + Bonferroni + Cohen's d
# ============================================================
print("📊 İstatistiksel Testler (Wilcoxon signed-rank)\n")

N_TEST    = len(MODEL_SIRASI) * 3   # 7 model × 3 gürültü = 21 test
ALPHA_DUZ = 0.05 / N_TEST           # Bonferroni

istat_kayitlar = []

for model_adi in MODEL_SIRASI:
    for pct in ['pct1','pct3','pct5']:
        shap_vals = np.array(tum_perturb[model_adi][pct]['shap']['spearman'])
        lime_vals = np.array(tum_perturb[model_adi][pct]['lime']['spearman'])

        diff = shap_vals - lime_vals
        try:
            stat, p = wilcoxon(diff)
        except:
            stat, p = np.nan, np.nan

        # Cohen's d
        ortalama_fark = np.mean(diff)
        std_fark      = np.std(diff, ddof=1)
        cohens_d      = ortalama_fark / std_fark if std_fark > 0 else np.nan

        istat_kayitlar.append({
            'Model'     : model_adi,
            'Gurultu'   : pct,
            'W_stat'    : round(stat, 2) if not np.isnan(stat) else np.nan,
            'p_raw'     : p,
            'p_bonf'    : min(p * N_TEST, 1.0),
            'Anlamli'   : p < ALPHA_DUZ,
            'Cohens_d'  : round(cohens_d, 3),
            'SHAP_mean' : round(np.mean(shap_vals), 4),
            'LIME_mean' : round(np.mean(lime_vals), 4),
        })

df_istat = pd.DataFrame(istat_kayitlar)
kaydet(df_istat, "df_istat_cicids.pkl")

print(f"Bonferroni α = 0.05 / {N_TEST} = {ALPHA_DUZ:.4f}\n")
print(f"{'Model':<22} {'Gürültü':>8} {'W':>8} {'p_bonf':>10} {'Anl':>5} {'d':>8}")
print("-" * 66)
for _, row in df_istat.iterrows():
    anlam = "✅" if row['Anlamli'] else "❌"
    print(f"  {row['Model']:<20} {row['Gurultu']:>8} "
          f"{row['W_stat']:>8.1f} {row['p_bonf']:>10.4f} "
          f"{anlam:>5} {row['Cohens_d']:>8.3f}")

print("\n✅ İstatistiksel testler tamamlandı!")

# ============================================================
# XAI-SCORE ÇERÇEVESİ
# ============================================================
print("\n\n📊 XAI-Score Çerçevesi\n")

# 1. Pertürbasyon kararlılığı (SHAP %5 Spearman ortalaması)
perturb = {m: np.mean(tum_perturb[m]['pct5']['shap']['spearman'])
           for m in MODEL_SIRASI}

# 2. Model varyasyonu (Spearman ortalaması)
varyans = (df_var.groupby('model_adi')['spearman']
           .mean().reindex(MODEL_SIRASI).to_dict())

# 3. Fidelity ΔF1@5
fidelity_raw = (df_fidelity[df_fidelity['n_maskele']==5]
                .groupby('model')['delta_f1']
                .mean().reindex(MODEL_SIRASI).to_dict())

# 4. Hesaplama süresi (XAI hücresinde ölçülen SHAP süreleri)
sure_ms = {}
for m in MODEL_SIRASI:
    xai = yukle(f"xai_cicids_{m}.pkl")
    sure_ms[m] = xai['shap_sure']
    del xai
gc.collect()

def normalize(d, inverse=False):
    vals = np.array(list(d.values()), dtype=float)
    mn, mx = vals.min(), vals.max()
    if mx == mn:
        return {k: 0.5 for k in d}
    normed = {k: (v - mn)/(mx - mn) for k, v in d.items()}
    if inverse:
        normed = {k: 1-v for k, v in normed.items()}
    return normed

perturb_n  = normalize(perturb)
varyans_n  = normalize(varyans)
fidelity_n = normalize(fidelity_raw)
sure_n     = normalize(sure_ms, inverse=True)

W = 0.25
xai_scores = {m: W*(perturb_n[m] + varyans_n[m] + fidelity_n[m] + sure_n[m])
              for m in MODEL_SIRASI}

df_xai = pd.DataFrame({
    'Model'      : MODEL_SIRASI,
    'Perturb'    : [round(perturb[m],4)     for m in MODEL_SIRASI],
    'Varyans'    : [round(varyans[m],4)     for m in MODEL_SIRASI],
    'Fidelity_5' : [round(fidelity_raw[m],4) for m in MODEL_SIRASI],
    'Sure_ms'    : [round(sure_ms[m],3)     for m in MODEL_SIRASI],
    'XAI_Score'  : [round(xai_scores[m],4) for m in MODEL_SIRASI],
}).sort_values('XAI_Score', ascending=False).reset_index(drop=True)

print("XAI-Score Sıralaması:")
print(df_xai.to_string(index=False))
kaydet(df_xai, "df_xai_score_cicids.pkl")
print("\n✅ XAI-Score tamamlandı!")

# ============================================================
# GÖRSELLEŞTİRME
# ============================================================
print("\n\n🎨 Görseller oluşturuluyor...\n")

# ── GÖRSEL 1: Ana 9 grafik (RQ1 + RQ2 + RQ3 + Fidelity) ─────────────────
fig, axes = plt.subplots(3, 3, figsize=(20, 15))
fig.suptitle('CIC-IDS-2017 — XAI Güvenilirlik Analizi',
             fontsize=16, fontweight='bold')

# 1a. RQ1: SHAP Spearman (gürültü seviyeleri)
ax = axes[0, 0]
x  = np.arange(3)
for m in MODEL_SIRASI:
    vals = [np.mean(tum_perturb[m][p]['shap']['spearman'])
            for p in ['pct1','pct3','pct5']]
    ax.plot(x, vals, 'o-', color=RENKLER[m], label=KISALTMA[m], linewidth=2)
ax.set_xticks(x)
ax.set_xticklabels(['%1','%3','%5'])
ax.set_title('RQ1: SHAP Pertürbasyon Kararlılığı', fontweight='bold')
ax.set_ylabel('Spearman r')
ax.set_ylim(0, 1.05)
ax.legend(fontsize=7)
ax.grid(alpha=0.3)

# 1b. RQ1: LIME Spearman
ax = axes[0, 1]
for m in MODEL_SIRASI:
    vals = [np.mean(tum_perturb[m][p]['lime']['spearman'])
            for p in ['pct1','pct3','pct5']]
    ax.plot(x, vals, 'o-', color=RENKLER[m], label=KISALTMA[m], linewidth=2)
ax.set_xticks(x)
ax.set_xticklabels(['%1','%3','%5'])
ax.set_title('RQ1: LIME Pertürbasyon Kararlılığı', fontweight='bold')
ax.set_ylabel('Spearman r')
ax.set_ylim(0, 0.5)
ax.legend(fontsize=7)
ax.grid(alpha=0.3)

# 1c. RQ1: SHAP vs LIME karşılaştırma (bar)
ax   = axes[0, 2]
shap_means = [np.mean(tum_perturb[m]['pct3']['shap']['spearman']) for m in MODEL_SIRASI]
lime_means = [np.mean(tum_perturb[m]['pct3']['lime']['spearman']) for m in MODEL_SIRASI]
xi    = np.arange(len(MODEL_SIRASI))
width = 0.35
ax.bar(xi - width/2, shap_means, width, label='SHAP', color='#2196F3', alpha=0.8)
ax.bar(xi + width/2, lime_means, width, label='LIME', color='#FF9800', alpha=0.8)
ax.set_xticks(xi)
ax.set_xticklabels([KISALTMA[m] for m in MODEL_SIRASI], fontsize=9)
ax.set_title('RQ1: SHAP vs LIME @%3 Gürültü', fontweight='bold')
ax.set_ylabel('Spearman r')
ax.legend()
ax.grid(alpha=0.3, axis='y')

# 2a. RQ2: Model Varyasyonu Spearman
ax = axes[1, 0]
sp_vals = [df_var[df_var['model_adi']==m]['spearman'].mean() for m in MODEL_SIRASI]
renkler = [RENKLER[m] for m in MODEL_SIRASI]
bars = ax.bar([KISALTMA[m] for m in MODEL_SIRASI], sp_vals, color=renkler, alpha=0.85)
ax.set_title('RQ2: Model Varyasyonu (Spearman)', fontweight='bold')
ax.set_ylabel('Spearman r')
ax.set_ylim(0, 1)
for bar, val in zip(bars, sp_vals):
    ax.text(bar.get_x()+bar.get_width()/2, val+0.01, f'{val:.3f}',
            ha='center', va='bottom', fontsize=8)
ax.grid(alpha=0.3, axis='y')

# 2b. RQ2: Jaccard@5 ve Jaccard@10
ax   = axes[1, 1]
j5   = [df_var[df_var['model_adi']==m]['jaccard_5'].mean()  for m in MODEL_SIRASI]
j10  = [df_var[df_var['model_adi']==m]['jaccard_10'].mean() for m in MODEL_SIRASI]
ax.bar(xi-width/2, j5,  width, label='Jaccard@5',  color='#3F51B5', alpha=0.8)
ax.bar(xi+width/2, j10, width, label='Jaccard@10', color='#E91E63', alpha=0.8)
ax.set_xticks(xi)
ax.set_xticklabels([KISALTMA[m] for m in MODEL_SIRASI], fontsize=9)
ax.set_title('RQ2: Jaccard Benzerliği', fontweight='bold')
ax.set_ylabel('Jaccard')
ax.legend()
ax.grid(alpha=0.3, axis='y')

# 2c. RQ3: F1 vs Spearman scatter
ax = axes[1, 2]
df_var2 = yukle("df_var_cicids.pkl")
for m in MODEL_SIRASI:
    alt = df_var2[df_var2['model_adi']==m]
    ax.scatter(alt['f1_macro'], alt['spearman'],
               color=RENKLER[m], label=KISALTMA[m], alpha=0.5, s=15)
r_all, _ = pearsonr(df_var2['f1_macro'], df_var2['spearman'])
ax.set_title(f'RQ3: F1 vs SHAP Kararlılığı (r={r_all:.3f})', fontweight='bold')
ax.set_xlabel('F1 Macro')
ax.set_ylabel('Spearman r')
ax.legend(fontsize=7)
ax.grid(alpha=0.3)

# 3a. Fidelity: ΔF1 kümülatif
ax = axes[2, 0]
for m in MODEL_SIRASI:
    alt  = df_fidelity[df_fidelity['model']==m].sort_values('n_maskele')
    ax.plot(alt['n_maskele'], alt['delta_f1'],
            'o-', color=RENKLER[m], label=KISALTMA[m], linewidth=2)
ax.set_title('Fidelity: Kümülatif SHAP Maskeleme', fontweight='bold')
ax.set_xlabel('Maskelenen Özellik Sayısı')
ax.set_ylabel('ΔF1 Macro')
ax.legend(fontsize=7)
ax.grid(alpha=0.3)

# 3b. Fidelity ΔF1@5 bar
ax   = axes[2, 1]
d5_vals = [df_fidelity[(df_fidelity['model']==m)&
                       (df_fidelity['n_maskele']==5)]['delta_f1'].values[0]
           for m in MODEL_SIRASI]
bars = ax.bar([KISALTMA[m] for m in MODEL_SIRASI], d5_vals,
              color=[RENKLER[m] for m in MODEL_SIRASI], alpha=0.85)
ax.set_title('Fidelity: ΔF1@5 Özellik', fontweight='bold')
ax.set_ylabel('ΔF1 Macro')
for bar, val in zip(bars, d5_vals):
    ax.text(bar.get_x()+bar.get_width()/2, val+0.01, f'{val:.3f}',
            ha='center', va='bottom', fontsize=8)
ax.grid(alpha=0.3, axis='y')

# 3c. XAI-Score bar chart
ax    = axes[2, 2]
sirali = df_xai['Model'].tolist()
skorlar= df_xai['XAI_Score'].tolist()
bars   = ax.barh(range(len(sirali)), skorlar,
                 color=[RENKLER[m] for m in sirali], alpha=0.85)
ax.set_yticks(range(len(sirali)))
ax.set_yticklabels([KISALTMA[m] for m in sirali])
ax.set_xlabel('XAI-Score')
ax.set_title('XAI-Score Sıralaması', fontweight='bold')
ax.set_xlim(0, 1)
for bar, val in zip(bars, skorlar):
    ax.text(val+0.01, bar.get_y()+bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9, fontweight='bold')
ax.grid(alpha=0.3, axis='x')

plt.tight_layout()
gorsel_yol = os.path.join(GORSEL_DIR, 'xai_cicids_tam_analiz.png')
plt.savefig(gorsel_yol, dpi=150, bbox_inches='tight')
plt.close()
print(f"✅ xai_cicids_tam_analiz.png kaydedildi")

# ── GÖRSEL 2: Radar Chart ─────────────────────────────────────────────────
kategoriler = ['Pertürbasyon\nKararlılık','Model\nVaryasyonu',
               'Fidelity','Hesaplama\nVerimliliği']
N_KAT  = len(kategoriler)
angles = np.linspace(0, 2*np.pi, N_KAT, endpoint=False).tolist()
angles += angles[:1]

fig, axes2 = plt.subplots(2, 4, figsize=(20, 10),
                           subplot_kw=dict(polar=True))
fig.suptitle('CIC-IDS-2017 — XAI Güvenilirlik Radar Chart',
             fontsize=13, fontweight='bold')
axes_flat = axes2.flatten()

for idx, model_adi in enumerate(MODEL_SIRASI):
    ax = axes_flat[idx]
    degerler = [perturb_n[model_adi], varyans_n[model_adi],
                fidelity_n[model_adi], sure_n[model_adi]]
    degerler += degerler[:1]
    renk = RENKLER[model_adi]
    ax.plot(angles, degerler, 'o-', linewidth=2, color=renk)
    ax.fill(angles, degerler, alpha=0.25, color=renk)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(kategoriler, fontsize=8)
    ax.set_ylim(0, 1)
    ax.set_title(f"{KISALTMA[model_adi]}\nXAI={xai_scores[model_adi]:.3f}",
                 fontweight='bold', fontsize=10, pad=12)
    ax.grid(alpha=0.3)

ax = axes_flat[7]
for model_adi in MODEL_SIRASI:
    degerler = [perturb_n[model_adi], varyans_n[model_adi],
                fidelity_n[model_adi], sure_n[model_adi]]
    degerler += degerler[:1]
    ax.plot(angles, degerler, 'o-', linewidth=1.5,
            color=RENKLER[model_adi], label=KISALTMA[model_adi], alpha=0.8)
    ax.fill(angles, degerler, alpha=0.05, color=RENKLER[model_adi])
ax.set_xticks(angles[:-1])
ax.set_xticklabels(kategoriler, fontsize=8)
ax.set_ylim(0, 1)
ax.set_title('Tüm Modeller', fontweight='bold', fontsize=10, pad=12)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=8)
ax.grid(alpha=0.3)

plt.tight_layout()
gorsel_yol = os.path.join(GORSEL_DIR, 'xai_cicids_radar_chart.png')
plt.savefig(gorsel_yol, dpi=150, bbox_inches='tight')
plt.close()
print(f"✅ xai_cicids_radar_chart.png kaydedildi")

# ── GÖRSEL 3: İstatistik özet ─────────────────────────────────────────────
fig, axes3 = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('CIC-IDS-2017 — İstatistiksel Analiz',
             fontsize=13, fontweight='bold')

# Wilcoxon Cohen's d
ax   = axes3[0]
pct_labels = ['%1','%3','%5']
x3   = np.arange(len(MODEL_SIRASI))
w3   = 0.25
for i, pct in enumerate(['pct1','pct3','pct5']):
    d_vals = [df_istat[(df_istat['Model']==m)&
                       (df_istat['Gurultu']==pct)]['Cohens_d'].values[0]
              for m in MODEL_SIRASI]
    ax.bar(x3 + (i-1)*w3, d_vals, w3, label=pct_labels[i], alpha=0.8)
ax.set_xticks(x3)
ax.set_xticklabels([KISALTMA[m] for m in MODEL_SIRASI])
ax.set_title("Cohen's d (SHAP vs LIME)", fontweight='bold')
ax.set_ylabel("Cohen's d")
ax.legend()
ax.grid(alpha=0.3, axis='y')

# SHAP vs LIME ortalama karşılaştırma
ax = axes3[1]
shap_avgs = [np.mean([np.mean(tum_perturb[m][p]['shap']['spearman'])
                      for p in ['pct1','pct3','pct5']]) for m in MODEL_SIRASI]
lime_avgs = [np.mean([np.mean(tum_perturb[m][p]['lime']['spearman'])
                      for p in ['pct1','pct3','pct5']]) for m in MODEL_SIRASI]
ax.bar(x3-width/2, shap_avgs, width, label='SHAP', color='#2196F3', alpha=0.8)
ax.bar(x3+width/2, lime_avgs, width, label='LIME', color='#FF9800', alpha=0.8)
ax.set_xticks(x3)
ax.set_xticklabels([KISALTMA[m] for m in MODEL_SIRASI])
ax.set_title('SHAP vs LIME Ortalama Kararlılık', fontweight='bold')
ax.set_ylabel('Spearman r')
ax.legend()
ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
gorsel_yol = os.path.join(GORSEL_DIR, 'xai_cicids_istatistik.png')
plt.savefig(gorsel_yol, dpi=150, bbox_inches='tight')
plt.close()
print(f"✅ xai_cicids_istatistik.png kaydedildi")

print("\n🎉 TÜM ANALİZLER TAMAMLANDI!")
print(f"\nCheckpoint dosyaları: {CHECKPOINT_DIR}")
print(f"Görsel dosyaları    : {GORSEL_DIR}")
print("\nKaydedilen görseller:")
print("  - xai_cicids_tam_analiz.png")
print("  - xai_cicids_radar_chart.png")
print("  - xai_cicids_istatistik.png")

In [ ]:
import os

GORSEL_DIR = r"str(CICIDS2017_DIR)"
CHECKPOINT_DIR = r"str(CICIDS2017_DIR)"

gozseller = [
    "xai_cicids_tam_analiz.png",
    "xai_cicids_radar_chart.png",
    "xai_cicids_istatistik.png",
    "xai_cicids_score_siralama.png",
]

print("=== GÖRSEL KONTROL ===")
for g in gozseller:
    yol = os.path.join(GORSEL_DIR, g)
    durum = "✅ VAR" if os.path.exists(yol) else "❌ YOK"
    print(f"  {durum} — {g}")

print("\n=== CHECKPOINT KONTROL ===")
checkpointler = [
    "veri_cicids.pkl",
    "df_var_cicids.pkl",
    "df_rq3_cicids.pkl",
    "df_fidelity_cicids.pkl",
    "df_istat_cicids.pkl",
    "df_xai_score_cicids.pkl",
    "tum_perturb_sonuclar_cicids.pkl",
]
for c in checkpointler:
    yol = os.path.join(CHECKPOINT_DIR, c)
    durum = "✅ VAR" if os.path.exists(yol) else "❌ YOK"
    print(f"  {durum} — {c}")

In [ ]:
import pickle, os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

CHECKPOINT_DIR = r"str(CICIDS2017_DIR)"
GORSEL_DIR     = r"str(CICIDS2017_DIR)"

RENKLER = {
    'RandomForest':'#2196F3','ExtraTrees':'#4CAF50','DecisionTree':'#FF9800',
    'XGBoost':'#F44336','LightGBM':'#9C27B0','CatBoost':'#00BCD4',
    'LogisticRegression':'#795548'
}
KISALTMA = {
    'RandomForest':'RF','ExtraTrees':'ET','DecisionTree':'DT',
    'XGBoost':'XGB','LightGBM':'LGBM','CatBoost':'CB','LogisticRegression':'LR'
}

with open(os.path.join(CHECKPOINT_DIR, 'df_xai_score_cicids.pkl'), 'rb') as f:
    df_xai = pickle.load(f)

print(df_xai)
print(df_xai.columns.tolist())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('CIC-IDS-2017 — XAI Score Analizi', fontsize=15, fontweight='bold', y=1.01)

# XAI_Score'a göre sırala
df_sorted = df_xai.sort_values('XAI_Score', ascending=True).reset_index(drop=True)

renkler_sirali = [RENKLER[m] for m in df_sorted['Model']]
kisaltmalar    = [KISALTMA[m] for m in df_sorted['Model']]

# ── Sol: XAI Score yatay bar ──────────────────────────────────────────
ax1 = axes[0]
bars = ax1.barh(kisaltmalar, df_sorted['XAI_Score'],
                color=renkler_sirali, edgecolor='white', linewidth=0.8, height=0.6)

for bar, val in zip(bars, df_sorted['XAI_Score']):
    ax1.text(val + 0.005, bar.get_y() + bar.get_height()/2,
             f'{val:.4f}', va='center', ha='left', fontsize=11, fontweight='bold')

ax1.set_xlabel('XAI Score', fontsize=12)
ax1.set_title('XAI Score Sıralaması\n(Perturb × Varyans × Fidelity × Hız)', fontsize=12)
ax1.set_xlim(0, 1.08)
ax1.axvline(x=df_xai['XAI_Score'].mean(), color='red', linestyle='--',
            alpha=0.7, linewidth=1.5, label=f'Ortalama: {df_xai["XAI_Score"].mean():.4f}')
ax1.legend(fontsize=10)
ax1.grid(axis='x', alpha=0.3)
ax1.spines[['top','right']].set_visible(False)

# ── Sağ: Bileşen karşılaştırma grouped bar ───────────────────────────
ax2 = axes[1]
df_s2 = df_xai.sort_values('XAI_Score', ascending=False).reset_index(drop=True)

metrikler   = ['Perturb', 'Varyans', 'Fidelity_5']
metrik_adi  = ['Pertürbasyon\n(SHAP@5%)', 'Model\nVaryasyonu', 'Fidelity\n@Top5']
n_model     = len(df_s2)
n_metrik    = len(metrikler)
x           = np.arange(n_model)
bar_w       = 0.22
offsets     = [-bar_w, 0, bar_w]
metric_colors = ['#1976D2', '#388E3C', '#F57C00']

for i, (met, adi, mc) in enumerate(zip(metrikler, metrik_adi, metric_colors)):
    vals = df_s2[met].values
    b = ax2.bar(x + offsets[i], vals, bar_w,
                label=adi, color=mc, alpha=0.85, edgecolor='white', linewidth=0.6)
    for rect, v in zip(b, vals):
        ax2.text(rect.get_x() + rect.get_width()/2, rect.get_height() + 0.012,
                 f'{v:.2f}', ha='center', va='bottom', fontsize=7.5, fontweight='bold')

ax2.set_xticks(x)
ax2.set_xticklabels([KISALTMA[m] for m in df_s2['Model']], fontsize=11)
ax2.set_ylabel('Skor', fontsize=12)
ax2.set_title('Model Bazında Bileşen Karşılaştırması\n(XAI Score\'a Göre Sıralı)', fontsize=12)
ax2.set_ylim(0, 1.15)
ax2.legend(fontsize=9, loc='upper right')
ax2.grid(axis='y', alpha=0.3)
ax2.spines[['top','right']].set_visible(False)

plt.tight_layout()
kayit_yolu = os.path.join(GORSEL_DIR, 'xai_cicids_score_siralama.png')
plt.savefig(kayit_yolu, dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Kaydedildi: {kayit_yolu}")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# LIME PARAMETRE ANALİZİ — CIC-IDS-2017
# Edge-IIoT ile aynı yapı: num_samples x num_features grid
# ════════════════════════════════════════════════════════════════════
import os, pickle, gc, time
import numpy as np, pandas as pd
import lime, lime.lime_tabular
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')

CHECKPOINT_DIR = r"str(CICIDS2017_DIR)"

def kaydet(nesne, dosya_adi):
    yol = os.path.join(CHECKPOINT_DIR, dosya_adi)
    with open(yol, 'wb') as f:
        pickle.dump(nesne, f)
    print(f"  💾 Kaydedildi: {dosya_adi}")

def yukle(dosya_adi):
    yol = os.path.join(CHECKPOINT_DIR, dosya_adi)
    with open(yol, 'rb') as f:
        return pickle.load(f)

def kontrol_et(dosya_adi):
    return os.path.exists(os.path.join(CHECKPOINT_DIR, dosya_adi))

def lime_to_vec(lime_dict, feature_names):
    vec = np.zeros(len(feature_names))
    for fname, fval in lime_dict.items():
        clean = fname.split('<=')[0].split('>')[0].split('<')[0].strip()
        for i, fn in enumerate(feature_names):
            if fn == clean:
                vec[i] = fval
                break
    return vec

def jaccard_top_k(v1, v2, k):
    t1 = set(np.argsort(np.abs(v1))[-k:])
    t2 = set(np.argsort(np.abs(v2))[-k:])
    return len(t1 & t2) / len(t1 | t2) if len(t1 | t2) > 0 else 0.0

MODEL_SIRASI = ['RandomForest','ExtraTrees','DecisionTree',
                'XGBoost','LightGBM','CatBoost','LogisticRegression']

# ── Veri ve modelleri yükle ──────────────────────────────────────────
print("Veri yükleniyor...")
veri = yukle("veri_cicids.pkl")
X_train = veri['X_train']
X_test  = veri['X_test']
feature_names = veri['feature_names']   # 78 özellik
class_names   = veri['class_names']
print(f"  X_train: {X_train.shape}, özellik: {len(feature_names)}")

# Seed 0 modellerini yükle (referans)
print("Referans modeller (seed 0) yükleniyor...")
seed0 = yukle("modeller_cicids_seed0.pkl")
ref_modeller = {m: seed0[m]['fold_models'][0] for m in MODEL_SIRASI}
del seed0; gc.collect()

# XAI sonuçlarını yükle
print("XAI sonuçları yükleniyor...")
xai_sonuclar = {}
for m in MODEL_SIRASI:
    xai_sonuclar[m] = yukle(f"xai_cicids_{m}.pkl")
print("  ✅ XAI yüklendi")

# LIME explainer
lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    X_train,
    feature_names=feature_names,
    class_names=class_names,
    discretize_continuous=True,
    random_state=42
)

# 500 örnek alt kümesini al
X_orneklem = veri.get('X_orneklem', X_test[:500])

# ── Parametre grid ───────────────────────────────────────────────────
NUM_SAMPLES_LIST  = [500, 1000, 2000, 5000]
NUM_FEATURES_LIST = [10, 20, 40, 78]      # 78 = CIC-IDS2017 özellik sayısı
N_ORNEK           = 100

print(f"\nLIME parametre taraması başlıyor...")
print(f"  num_samples  : {NUM_SAMPLES_LIST}")
print(f"  num_features : {NUM_FEATURES_LIST}")
print(f"  Örnek sayısı : {N_ORNEK}/kombinasyon")
print(f"  Toplam       : {len(MODEL_SIRASI)} × {len(NUM_SAMPLES_LIST)} × {len(NUM_FEATURES_LIST)} = "
      f"{len(MODEL_SIRASI)*len(NUM_SAMPLES_LIST)*len(NUM_FEATURES_LIST)} kombinasyon\n")

# ── Checkpoint — kaldığı yerden devam ───────────────────────────────
if kontrol_et("lime_param_sonuclar_cicids.pkl"):
    lime_param_sonuclar = yukle("lime_param_sonuclar_cicids.pkl")
    print(f"Mevcut checkpoint yüklendi: {len(lime_param_sonuclar)} kayıt")
else:
    lime_param_sonuclar = []

tamamlanan = set(
    (r['model'], r['num_samples'], r['num_features'])
    for r in lime_param_sonuclar
)

np.random.seed(42)
alt_idx = np.random.choice(len(X_orneklem), size=N_ORNEK, replace=False)
X_alt   = X_orneklem[alt_idx]

# ── Ana döngü ────────────────────────────────────────────────────────
for model_adi in MODEL_SIRASI:
    model     = ref_modeller[model_adi]
    shap_orig = xai_sonuclar[model_adi]['shap_values'][alt_idx]

    for ns in NUM_SAMPLES_LIST:
        for nf in NUM_FEATURES_LIST:

            if (model_adi, ns, nf) in tamamlanan:
                print(f"  ♻  {model_adi:<22} ns={ns:>4}  nf={nf:>2}  — atlandı")
                continue

            t0      = time.time()
            sp_list = []

            for i in range(N_ORNEK):
                exp = lime_explainer.explain_instance(
                    X_alt[i],
                    model.predict_proba,
                    num_features=nf,
                    num_samples=ns
                )
                lime_vec = lime_to_vec(dict(exp.as_list()), feature_names)
                sp, _    = spearmanr(shap_orig[i], lime_vec)
                sp_list.append(sp)

            sure  = (time.time() - t0) / N_ORNEK * 1000
            spear = float(np.nanmean(sp_list))

            lime_param_sonuclar.append({
                'model'       : model_adi,
                'num_samples' : ns,
                'num_features': nf,
                'spearman'    : round(spear, 4),
                'sure_ms'     : round(sure, 1),
            })
            kaydet(lime_param_sonuclar, "lime_param_sonuclar_cicids.pkl")

            print(f"  ✅ {model_adi:<22} ns={ns:>4}  nf={nf:>2}  "
                  f"Spearman={spear:.4f}  ({sure:.0f} ms/örnek)")

# ── Özet ─────────────────────────────────────────────────────────────
df_lime_param = pd.DataFrame(lime_param_sonuclar)
kaydet(df_lime_param, "df_lime_param_cicids.pkl")

print("\n✅ Tamamlandı! df_lime_param_cicids.pkl kaydedildi.")
print("\nOrtalama Spearman (num_samples × num_features):")
print(df_lime_param.groupby(['num_samples','num_features'])['spearman'].mean().unstack())

In [ ]:
print(f"X_scaled shape: {veri['X_scaled'].shape}")
print(f"y shape: {veri['y'].shape}")
print(f"Sınıflar: {veri['sinif_isimleri']}")

# XAI dosyasının yapısını da kontrol et
xai_rf = yukle("xai_cicids_RandomForest.pkl")
print(f"\nXAI keys: {xai_rf.keys()}")
print(f"shap_values shape: {xai_rf['shap_values'].shape}")

In [ ]:
print(f"Liste uzunluğu: {len(seed0)}")
print(f"İlk eleman tipi: {type(seed0[0])}")
ilk = seed0[0]
if isinstance(ilk, dict):
    print(f"Keys: {ilk.keys()}")
else:
    print(ilk)

In [ ]:
# ── Veri yükle ───────────────────────────────────────────────────────
veri = yukle("veri_cicids.pkl")
X_scaled      = veri['X_scaled']
y             = veri['y']
feature_names = veri['feature_names']
class_names   = list(veri['sinif_isimleri'])
print(f"X: {X_scaled.shape}, özellik: {len(feature_names)}")

# ── Referans modelleri yükle (seed0, fold0) ──────────────────────────
seed0 = yukle("modeller_cicids_seed0.pkl")
ref_modeller = {}
for kayit in seed0:
    if kayit['fold'] == 0:
        ref_modeller[kayit['model_adi']] = kayit['model_obj']
del seed0; gc.collect()
print(f"Referans modeller: {list(ref_modeller.keys())}")

# ── XAI yükle ────────────────────────────────────────────────────────
xai_sonuclar = {}
for m in MODEL_SIRASI:
    xai_sonuclar[m] = yukle(f"xai_cicids_{m}.pkl")
print("XAI yüklendi ✅")

# ── XAI'da kullanılan 500 örneği reproduce et ────────────────────────
np.random.seed(42)
orneklem_idx = np.random.choice(len(X_scaled), size=500, replace=False)
X_orneklem   = X_scaled[orneklem_idx]

# ── LIME explainer ───────────────────────────────────────────────────
lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    X_scaled,
    feature_names=feature_names,
    class_names=class_names,
    discretize_continuous=True,
    random_state=42
)

# ── Parametre grid ───────────────────────────────────────────────────
NUM_SAMPLES_LIST  = [500, 1000, 2000, 5000]
NUM_FEATURES_LIST = [10, 20, 40, 78]
N_ORNEK           = 100

print(f"\nLIME parametre taraması başlıyor...")
print(f"  Toplam: {len(MODEL_SIRASI)} × {len(NUM_SAMPLES_LIST)} × {len(NUM_FEATURES_LIST)} = "
      f"{len(MODEL_SIRASI)*len(NUM_SAMPLES_LIST)*len(NUM_FEATURES_LIST)} kombinasyon\n")

# ── Checkpoint ───────────────────────────────────────────────────────
if kontrol_et("lime_param_sonuclar_cicids.pkl"):
    lime_param_sonuclar = yukle("lime_param_sonuclar_cicids.pkl")
    print(f"Checkpoint yüklendi: {len(lime_param_sonuclar)} kayıt")
else:
    lime_param_sonuclar = []

tamamlanan = set(
    (r['model'], r['num_samples'], r['num_features'])
    for r in lime_param_sonuclar
)

np.random.seed(42)
alt_idx = np.random.choice(len(X_orneklem), size=N_ORNEK, replace=False)
X_alt   = X_orneklem[alt_idx]

# ── Ana döngü ────────────────────────────────────────────────────────
for model_adi in MODEL_SIRASI:
    model     = ref_modeller[model_adi]
    shap_orig = xai_sonuclar[model_adi]['shap_values'][alt_idx]

    for ns in NUM_SAMPLES_LIST:
        for nf in NUM_FEATURES_LIST:

            if (model_adi, ns, nf) in tamamlanan:
                print(f"  ♻  {model_adi:<22} ns={ns:>4}  nf={nf:>2}  — atlandı")
                continue

            t0      = time.time()
            sp_list = []

            for i in range(N_ORNEK):
                exp = lime_explainer.explain_instance(
                    X_alt[i],
                    model.predict_proba,
                    num_features=nf,
                    num_samples=ns
                )
                lime_vec = lime_to_vec(dict(exp.as_list()), feature_names)
                sp, _    = spearmanr(shap_orig[i], lime_vec)
                sp_list.append(sp)

            sure  = (time.time() - t0) / N_ORNEK * 1000
            spear = float(np.nanmean(sp_list))

            lime_param_sonuclar.append({
                'model'       : model_adi,
                'num_samples' : ns,
                'num_features': nf,
                'spearman'    : round(spear, 4),
                'sure_ms'     : round(sure, 1),
            })
            kaydet(lime_param_sonuclar, "lime_param_sonuclar_cicids.pkl")
            print(f"  ✅ {model_adi:<22} ns={ns:>4}  nf={nf:>2}  "
                  f"Spearman={spear:.4f}  ({sure:.0f} ms/örnek)")

# ── Özet ─────────────────────────────────────────────────────────────
df_lime_param = pd.DataFrame(lime_param_sonuclar)
kaydet(df_lime_param, "df_lime_param_cicids.pkl")
print("\n✅ Tamamlandı!")
print(df_lime_param.groupby(['num_samples','num_features'])['spearman'].mean().unstack())

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd

GORSEL_DIR = r"str(CICIDS2017_DIR)"

df_lp = pd.DataFrame(lime_param_sonuclar)

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
fig.suptitle('CIC-IDS-2017 — LIME Parametre Analizi\n(SHAP ile Spearman Korelasyonu)',
             fontsize=14, fontweight='bold')

for ax, model_adi in zip(axes.flatten(), MODEL_SIRASI):
    df_m = df_lp[df_lp['model'] == model_adi]
    pivot = df_m.pivot(index='num_samples', columns='num_features', values='spearman')

    im = ax.imshow(pivot.values, cmap='RdYlGn', vmin=-0.2, vmax=0.2, aspect='auto')

    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, fontsize=8)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index, fontsize=8)
    ax.set_xlabel('num_features', fontsize=8)
    ax.set_ylabel('num_samples', fontsize=8)
    ax.set_title(f'{KISALTMA[model_adi]}', fontsize=11, fontweight='bold',
                 color=RENKLER[model_adi])

    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            val = pivot.values[i, j]
            ax.text(j, i, f'{val:.3f}', ha='center', va='center',
                    fontsize=7.5, fontweight='bold',
                    color='black' if abs(val) < 0.12 else 'white')

    fig.colorbar(im, ax=ax, shrink=0.8)

# Son eksen boş — genel özet yaz
axes[1][3].axis('off')
genel_ort = df_lp['spearman'].mean()
genel_std = df_lp['spearman'].std()
genel_min = df_lp['spearman'].min()
genel_max = df_lp['spearman'].max()

ozet_text = (
    f"GENEL ÖZET\n\n"
    f"Ortalama r : {genel_ort:.4f}\n"
    f"Std         : {genel_std:.4f}\n"
    f"Min         : {genel_min:.4f}\n"
    f"Max         : {genel_max:.4f}\n\n"
    f"Tüm kombinasyonlarda\n"
    f"r ≈ 0 → LIME parametreleri\n"
    f"SHAP uyumunu artırmıyor.\n\n"
    f"Edge-IIoT ile tutarlı."
)
axes[1][3].text(0.1, 0.5, ozet_text, transform=axes[1][3].transAxes,
                fontsize=10, verticalalignment='center',
                bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
kayit_yolu = os.path.join(GORSEL_DIR, 'lime_cicids_parametre_analizi.png')
plt.savefig(kayit_yolu, dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Kaydedildi: {kayit_yolu}")